In [ ]:
# Purpose: imports and small setup
import logging
logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')


In [ ]:
# Purpose: plotting / figure generation
# Reproducibility helper added by tools/add_repro_cell.py
import os, sys, random, json
from datetime import datetime
import numpy as np
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
# Optional: set torch deterministic flags if torch is available
try:
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # deterministic settings (may affect performance)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
except Exception:
    pass
# Record environment info for reproducibility
info = dict(
    python=sys.executable,
    python_version='.'.join(map(str, sys.version_info[:3])),
    numpy=np.__version__,
    seed=seed,
    timestamp=str(datetime.utcnow())
)
# Add optional package checks
try:
    import pkg_resources
    pkgs=['nbformat','pandas','scipy','sklearn','matplotlib','seaborn']
    pv = {p: pkg_resources.get_distribution(p).version for p in pkgs if pkg_resources.get_distribution(p) is not None}
    info['packages'] = pv
except Exception:
    pass
logger.info('REPRO: ' + json.dumps(info))


In [2]:
# Purpose: imports and small setup
from google.colab import drive, userdata
from pathlib import Path
import sys
import json

# (commented) drive.mount("/content/drive")

# (sanitized path) PROJECT_ROOT = Path("PROJECT_ROOT")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root missing: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

RUN_ID = "pilot_001"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_DIR = PROJECT_ROOT / "prompts"
CONFIGS_DIR = PROJECT_ROOT / "configs"

ANTHROPIC_KEY = userdata.get("Anthropic")
OPENAI_KEY = userdata.get("OPENAI_API_KEY")
GROQ_KEY = userdata.get("Groq")
for name, value in [("Anthropic", ANTHROPIC_KEY), ("OpenAI", OPENAI_KEY), ("Groq", GROQ_KEY)]:
    if not value:
        raise RuntimeError(f"Secret not loaded: {name}")

logger.info(f"PROJECT_ROOT: {PROJECT_ROOT}")
logger.info(f"RUN_DIR:      {RUN_DIR}")
logger.info(f"Secrets loaded: Anthropic, OpenAI, Groq")


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Dr Dacon ICDM/SeamSFF-main
RUN_DIR:      /content/drive/MyDrive/Dr Dacon ICDM/SeamSFF-main/runs/pilot_001
Secrets loaded: Anthropic, OpenAI, Groq


In [3]:
# Purpose: data processing / analysis step
!pip install -q anthropic openai groq

CONFIG_PATH = CONFIGS_DIR / "project_config.json"
with CONFIG_PATH.open() as f:
    PROJECT_CONFIG = json.load(f)

POOL_FILES = {
    "python_debugging":              PROJECT_ROOT / "python_domain"   / "selected" / "python_domain_selected.jsonl",
    "clinical_text_interpretation":  PROJECT_ROOT / "clinical_domain" / "selected" / "clinical_domain_selected.jsonl",
    "historical_document_synthesis": PROJECT_ROOT / "history_domain"  / "selected" / "history_domain_selected_180.jsonl",
}

TASK_POOLS = {}
for domain, path in POOL_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Pool missing: {path}")
    with path.open() as f:
        TASK_POOLS[domain] = [json.loads(line) for line in f if line.strip()]
    logger.info(f"{domain:40s}  {len(TASK_POOLS[domain]):3d} tasks  from  {path.name}")

PROMPT_FILES = {
    "seamless_agent":       "seamless_control_agent.md",
    "sff_agent":            "sff_treatment_agent.md",
    "persona_novice":       "persona_novice.md",
    "persona_intermediate": "persona_intermediate.md",
    "persona_expert":       "persona_expert.md",
    "judge":                "judge_quality_scorer.md",
    "state_tagger":         "state_tagger.md",
}

PROMPTS = {}
for name, filename in PROMPT_FILES.items():
    path = PROMPTS_DIR / filename
    PROMPTS[name] = path.read_text(encoding="utf-8")
    logger.info(f"  loaded prompt: {name:24s} {len(PROMPTS[name]):5d} chars")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.6/838.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.4 MB/s eta 0:00:00
python_debugging                          180 tasks  from  python_domain_selected.jsonl
clinical_text_interpretation              240 tasks  from  clinical_domain_selected.jsonl
historical_document_synthesis             180 tasks  from  history_domain_selected_180.jsonl
  loaded prompt: seamless_agent            3685 chars
  loaded prompt: sff_agent                 8139 chars
  loaded prompt: persona_novice            3044 chars
  loaded prompt: persona_intermediate      3154 chars
  loaded prompt: persona_expert            3153 chars
  loaded prompt: judge                     4589 chars
  loaded prompt: state_tagger              2874 chars


In [4]:
# Purpose: imports and small setup
import re

PROMPT_REQUIRED_VARS = {
    "seamless_agent":       {"session_id", "domain", "persona_level", "difficulty_tier", "benchmark_source", "benchmark_task_id", "task_prompt", "task_context"},
    "sff_agent":            {"session_id", "domain", "persona_level", "difficulty_tier", "benchmark_source", "benchmark_task_id", "task_prompt", "task_context"},
    "persona_novice":       {"session_id", "domain", "difficulty_tier", "benchmark_source", "benchmark_task_id", "prior_turns"},
    "persona_intermediate": {"session_id", "domain", "difficulty_tier", "benchmark_source", "benchmark_task_id", "prior_turns"},
    "persona_expert":       {"session_id", "domain", "difficulty_tier", "benchmark_source", "benchmark_task_id", "prior_turns"},
    "judge":                {"session_id", "domain", "difficulty_tier", "benchmark_source", "benchmark_task_id", "task_prompt", "ground_truth", "frozen_rubric", "required_output_schema", "test_pass_rate", "local_execution_logs", "agent_response"},
    "state_tagger":         {"session_id", "domain", "difficulty_tier", "prior_turns"},
}

PLACEHOLDER_RE = re.compile(r"\{\{([a-z_]+)\}\}")


def render_prompt(prompt_name: str, variables: dict) -> str:
    if prompt_name not in PROMPTS:
        raise KeyError(f"Unknown prompt: {prompt_name}")
    template = PROMPTS[prompt_name]
    required = PROMPT_REQUIRED_VARS[prompt_name]
    missing = required - set(variables.keys())
    if missing:
        raise ValueError(f"Missing variables for {prompt_name}: {sorted(missing)}")
    found_in_template = set(PLACEHOLDER_RE.findall(template))
    unexpected_required = required - found_in_template
    if unexpected_required:
        raise ValueError(f"{prompt_name} declares vars not present in template: {sorted(unexpected_required)}")

    def stringify(value):
        if isinstance(value, str):
            return value
        return json.dumps(value, ensure_ascii=False, indent=2)

    rendered = template
    for var in found_in_template:
        if var in variables:
            rendered = rendered.replace(f"{{{{{var}}}}}", stringify(variables[var]))
    remaining = PLACEHOLDER_RE.findall(rendered)
    if remaining:
        raise ValueError(f"Unfilled placeholders after rendering {prompt_name}: {sorted(set(remaining))}")
    return rendered


sample_task = TASK_POOLS["python_debugging"][0]
test_vars = {
    "session_id":        "smoke_test_001",
    "domain":            "python_debugging",
    "persona_level":     "novice",
    "difficulty_tier":   sample_task["difficulty_tier"],
    "benchmark_source":  sample_task["benchmark_source"],
    "benchmark_task_id": sample_task["task_id"],
    "task_prompt":       sample_task["task_prompt"],
    "task_context":      sample_task.get("context", ""),
}

rendered_seamless = render_prompt("seamless_agent", test_vars)
rendered_sff = render_prompt("sff_agent", test_vars)

logger.info(f"seamless rendered: {len(rendered_seamless)} chars (template was {len(PROMPTS['seamless_agent'])})")
logger.info(f"sff rendered:      {len(rendered_sff)} chars (template was {len(PROMPTS['sff_agent'])})")
logger.info()
logger.info("First 600 chars of rendered SFF prompt:")
logger.info("-" * 60)
logger.info(rendered_sff[:600])


ValueError: Unfilled placeholders after rendering seamless_agent: ['prior_turns']

In [ ]:
# Purpose: imports and small setup
import time
import random
import anthropic
import openai
import groq

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
openai_client = openai.OpenAI(api_key=OPENAI_KEY)
groq_client = groq.Groq(api_key=GROQ_KEY)

MODEL_HAIKU = "claude-haiku-4-5"
MODEL_GPT35 = "gpt-3.5-turbo"
MODEL_LLAMA = "llama-3.3-70b-versatile"

MAX_RETRIES = 4
BASE_BACKOFF = 1.5

ANTHROPIC_RETRYABLE = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError)
OPENAI_RETRYABLE = (openai.RateLimitError, openai.APIConnectionError, openai.InternalServerError)
GROQ_RETRYABLE = (groq.RateLimitError, groq.APIConnectionError, groq.InternalServerError)


def _backoff(attempt: int) -> float:
    return (BASE_BACKOFF ** attempt) + random.uniform(0, 0.5)


def call_anthropic(system: str, user: str, max_tokens: int = 1500) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = anthropic_client.messages.create(
                model=MODEL_HAIKU,
                max_tokens=max_tokens,
                system=system,
                messages=[{"role": "user", "content": user}],
            )
            return {
                "text": r.content[0].text,
                "tokens_in": r.usage.input_tokens,
                "tokens_out": r.usage.output_tokens,
                "model": MODEL_HAIKU,
                "stop_reason": r.stop_reason,
            }
        except ANTHROPIC_RETRYABLE as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_backoff(attempt))


def call_openai(system: str, user: str, max_tokens: int = 1500) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = openai_client.chat.completions.create(
                model=MODEL_GPT35,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            )
            return {
                "text": r.choices[0].message.content,
                "tokens_in": r.usage.prompt_tokens,
                "tokens_out": r.usage.completion_tokens,
                "model": MODEL_GPT35,
                "stop_reason": r.choices[0].finish_reason,
            }
        except OPENAI_RETRYABLE as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_backoff(attempt))


def call_groq(system: str, user: str, max_tokens: int = 1500) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = groq_client.chat.completions.create(
                model=MODEL_LLAMA,
                max_tokens=max_tokens,
                temperature=0.0,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            )
            return {
                "text": r.choices[0].message.content,
                "tokens_in": r.usage.prompt_tokens,
                "tokens_out": r.usage.completion_tokens,
                "model": MODEL_LLAMA,
                "stop_reason": r.choices[0].finish_reason,
            }
        except GROQ_RETRYABLE as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_backoff(attempt))


smoke_anthropic = call_anthropic(system="You are a test.", user="Reply with: ok")
smoke_openai = call_openai(system="You are a test.", user="Reply with: ok")
smoke_groq = call_groq(system="You are a test.", user="Reply with: ok")

for name, r in [("anthropic", smoke_anthropic), ("openai", smoke_openai), ("groq", smoke_groq)]:
    logger.info(f"{name:10s}  model={r['model']:32s}  in={r['tokens_in']:3d}  out={r['tokens_out']:3d}  stop={str(r['stop_reason']):12s}  text={r['text'].strip()!r}")


In [ ]:
# Purpose: imports and small setup
import re

JSON_FENCE_RE = re.compile(r"```(?:json)?\s*(\{.*?\})\s*```", re.DOTALL)
JSON_OBJECT_RE = re.compile(r"(\{.*\})", re.DOTALL)


def extract_json(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = JSON_FENCE_RE.search(text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    m = JSON_OBJECT_RE.search(text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    raise ValueError(f"No valid JSON object found in response. First 300 chars: {text[:300]!r}")


def call_with_json(call_fn, system: str, user: str, max_tokens: int = 1500, required_keys: set = None, max_attempts: int = 3) -> dict:
    last_error = None
    last_raw = None
    for attempt in range(max_attempts):
        response = call_fn(system=system, user=user, max_tokens=max_tokens)
        last_raw = response["text"]
        try:
            parsed = extract_json(last_raw)
        except ValueError as e:
            last_error = str(e)
            continue
        if required_keys and not required_keys.issubset(parsed.keys()):
            last_error = f"Missing required keys: {sorted(required_keys - set(parsed.keys()))}"
            continue
        return {
            "parsed": parsed,
            "raw_text": last_raw,
            "tokens_in": response["tokens_in"],
            "tokens_out": response["tokens_out"],
            "model": response["model"],
            "stop_reason": response["stop_reason"],
            "attempts": attempt + 1,
        }
    raise RuntimeError(f"JSON extraction failed after {max_attempts} attempts. Last error: {last_error}. Last raw (first 400): {last_raw[:400] if last_raw else None!r}")


cases = [
    ('clean', '{"state_label": "S2_INITIAL_ORIENTATION", "score": 0.85}'),
    ('fenced', '```json\n{"state_label": "S4_SOCRATIC_PROMPT", "score": 0.5}\n```'),
# (sanitized path)     ('prefixed', 'Here is my responsPROJECT_ROOT"state_label": "S6_USER_ACCEPT", "score": 1.0}'),
    ('with_trailing', '{"state_label": "S9_TASK_COMPLETE"}\nThat is my answer.'),
]

for name, text in cases:
    result = extract_json(text)
    logger.info(f"{name:18s}  parsed={result}")


In [ ]:
# Purpose: execution-based scoring of code solutions
AGENT_REQUIRED_KEYS = {"state_label", "visible_response", "policy_action"}
PERSONA_REQUIRED_KEYS = {"state_label", "visible_response", "policy_action"}
JUDGE_REQUIRED_KEYS = {"session_id", "domain_raw_score"}
STATE_TAGGER_REQUIRED_KEYS = {"state_sequence", "terminal_state"}

PERSONA_PROMPT_BY_LEVEL = {
    "novice":       "persona_novice",
    "intermediate": "persona_intermediate",
    "expert":       "persona_expert",
}


def run_agent_turn(condition: str, task: dict, persona_level: str, session_id: str, task_context: str = "") -> dict:
    prompt_name = "sff_agent" if condition == "sff" else "seamless_agent"
    variables = {
        "session_id":        session_id,
        "domain":            task["domain"],
        "persona_level":     persona_level,
        "difficulty_tier":   task["difficulty_tier"],
        "benchmark_source":  task["benchmark_source"],
        "benchmark_task_id": task["task_id"],
        "task_prompt":       task["task_prompt"],
        "task_context":      task_context or task.get("context", "") or "",
    }
    rendered = render_prompt(prompt_name, variables)
    return call_with_json(
        call_anthropic,
        system=rendered,
        user="Produce the next agent turn following the system policy. Output JSON only.",
        max_tokens=1500,
        required_keys=AGENT_REQUIRED_KEYS,
    )


def run_persona_turn(persona_level: str, task: dict, prior_turns: list, session_id: str) -> dict:
    prompt_name = PERSONA_PROMPT_BY_LEVEL[persona_level]
    variables = {
        "session_id":        session_id,
        "domain":            task["domain"],
        "difficulty_tier":   task["difficulty_tier"],
        "benchmark_source":  task["benchmark_source"],
        "benchmark_task_id": task["task_id"],
        "prior_turns":       prior_turns,
    }
    rendered = render_prompt(prompt_name, variables)
    return call_with_json(
        call_openai,
        system=rendered,
        user="Produce the next learner turn following the system policy. Output JSON only.",
        max_tokens=1000,
        required_keys=PERSONA_REQUIRED_KEYS,
    )


def run_judge(task: dict, agent_response: str, session_id: str, test_pass_rate=None, local_execution_logs: str = "") -> dict:
    variables = {
        "session_id":             session_id,
        "domain":                 task["domain"],
        "difficulty_tier":        task["difficulty_tier"],
        "benchmark_source":       task["benchmark_source"],
        "benchmark_task_id":      task["task_id"],
        "task_prompt":            task["task_prompt"],
        "ground_truth":           task["ground_truth"],
        "frozen_rubric":          task.get("rubric", ""),
        "required_output_schema": task.get("task_type", ""),
        "test_pass_rate":         test_pass_rate if test_pass_rate is not None else "not_applicable",
        "local_execution_logs":   local_execution_logs or "not_applicable",
        "agent_response":         agent_response,
    }
    rendered = render_prompt("judge", variables)
    return call_with_json(
        call_groq,
        system=rendered,
        user="Audit the final session and produce the blinded score. Output JSON only.",
        max_tokens=2000,
        required_keys=JUDGE_REQUIRED_KEYS,
    )


def run_state_tagger(task: dict, prior_turns: list, session_id: str) -> dict:
    variables = {
        "session_id":      session_id,
        "domain":          task["domain"],
        "difficulty_tier": task["difficulty_tier"],
        "prior_turns":     prior_turns,
    }
    rendered = render_prompt("state_tagger", variables)
    return call_with_json(
        call_groq,
        system=rendered,
        user="Re-tag the session blind. Output JSON only.",
        max_tokens=2000,
        required_keys=STATE_TAGGER_REQUIRED_KEYS,
    )


logger.info("Role functions defined: run_agent_turn, run_persona_turn, run_judge, run_state_tagger")


In [ ]:
# Purpose: data processing / analysis step
SMOKE_TASK = TASK_POOLS["python_debugging"][0]
SMOKE_SESSION_ID = "smoke_test_session_001"

logger.info("=" * 60)
logger.info(f"Smoke task: {SMOKE_TASK['task_id']}  ({SMOKE_TASK['benchmark_source']}, {SMOKE_TASK['difficulty_tier']})")
logger.info(f"Task prompt preview: {SMOKE_TASK['task_prompt'][:120]!r}")
logger.info("=" * 60)

logger.info("\n--- SFF agent turn (Haiku 4.5) ---")
agent_result = run_agent_turn(
    condition="sff",
    task=SMOKE_TASK,
    persona_level="novice",
    session_id=SMOKE_SESSION_ID,
)
logger.info(f"  attempts:        {agent_result['attempts']}")
logger.info(f"  tokens in/out:   {agent_result['tokens_in']} / {agent_result['tokens_out']}")
logger.info(f"  state_label:     {agent_result['parsed'].get('state_label')!r}")
logger.info(f"  policy_action:   {agent_result['parsed'].get('policy_action')!r}")
logger.info(f"  visible_response (first 200): {agent_result['parsed'].get('visible_response', '')[:200]!r}")

prior_turns_after_agent = [{
    "turn_index": 0,
    "speaker": "agent",
    "visible_response": agent_result['parsed']['visible_response'],
}]

logger.info("\n--- Persona turn (GPT-3.5, novice) ---")
persona_result = run_persona_turn(
    persona_level="novice",
    task=SMOKE_TASK,
    prior_turns=prior_turns_after_agent,
    session_id=SMOKE_SESSION_ID,
)
logger.info(f"  attempts:        {persona_result['attempts']}")
logger.info(f"  tokens in/out:   {persona_result['tokens_in']} / {persona_result['tokens_out']}")
logger.info(f"  state_label:     {persona_result['parsed'].get('state_label')!r}")
logger.info(f"  policy_action:   {persona_result['parsed'].get('policy_action')!r}")
logger.info(f"  visible_response (first 200): {persona_result['parsed'].get('visible_response', '')[:200]!r}")

prior_turns_after_persona = prior_turns_after_agent + [{
    "turn_index": 1,
    "speaker": "learner",
    "visible_response": persona_result['parsed']['visible_response'],
}]

logger.info("\n--- Blinded state tagger (Llama 3.3 70B) ---")
tagger_result = run_state_tagger(
    task=SMOKE_TASK,
    prior_turns=prior_turns_after_persona,
    session_id=SMOKE_SESSION_ID,
)
logger.info(f"  attempts:        {tagger_result['attempts']}")
logger.info(f"  tokens in/out:   {tagger_result['tokens_in']} / {tagger_result['tokens_out']}")
logger.info(f"  terminal_state:  {tagger_result['parsed'].get('terminal_state')!r}")
logger.info(f"  state_sequence ({len(tagger_result['parsed'].get('state_sequence', []))} turns):")
for s in tagger_result['parsed'].get('state_sequence', []):
    logger.info(f"    turn {s.get('turn_index')}: {s.get('speaker')} -> {s.get('state')}")

logger.info("\n--- Blinded judge (Llama 3.3 70B) ---")
judge_result = run_judge(
    task=SMOKE_TASK,
    agent_response=agent_result['parsed']['visible_response'],
    session_id=SMOKE_SESSION_ID,
)
logger.info(f"  attempts:        {judge_result['attempts']}")
logger.info(f"  tokens in/out:   {judge_result['tokens_in']} / {judge_result['tokens_out']}")
logger.info(f"  domain_raw_score: {judge_result['parsed'].get('domain_raw_score')}")


In [ ]:
# Purpose: data processing / analysis step
TURN_CAP = 8
TERMINAL_STATES = {"S9_TASK_COMPLETE", "S10_FAILURE"}


def run_one_session(condition: str, persona_level: str, task: dict, session_id: str) -> dict:
    turns = []
    prior_turns_view = []
    total_tokens_in = 0
    total_tokens_out = 0
    failure_reason = None

    for turn_index in range(TURN_CAP):
        try:
            if turn_index % 2 == 0:
                role = "agent"
                result = run_agent_turn(condition=condition, task=task, persona_level=persona_level, session_id=session_id)
            else:
                role = "learner"
                result = run_persona_turn(persona_level=persona_level, task=task, prior_turns=prior_turns_view, session_id=session_id)
        except Exception as e:
            failure_reason = f"{role}_call_failed_turn_{turn_index}: {type(e).__name__}: {e}"
            break

        parsed = result["parsed"]
        turn_record = {
            "turn_index":       turn_index,
            "speaker":          role,
            "state_label":      parsed.get("state_label"),
            "policy_action":    parsed.get("policy_action"),
            "visible_response": parsed.get("visible_response", ""),
            "metadata":         parsed.get("metadata", {}),
            "model":            result["model"],
            "tokens_in":        result["tokens_in"],
            "tokens_out":       result["tokens_out"],
            "attempts":         result["attempts"],
            "raw_text":         result["raw_text"],
        }
        turns.append(turn_record)
        prior_turns_view.append({
            "turn_index":       turn_index,
            "speaker":          role,
            "visible_response": parsed.get("visible_response", ""),
        })
        total_tokens_in += result["tokens_in"]
        total_tokens_out += result["tokens_out"]

        if parsed.get("state_label") in TERMINAL_STATES:
            break

    final_agent_response = next(
        (t["visible_response"] for t in reversed(turns) if t["speaker"] == "agent"),
        "",
    )

    return {
        "session_id":           session_id,
        "condition":            condition,
        "persona_level":        persona_level,
        "domain":               task["domain"],
        "difficulty_tier":      task["difficulty_tier"],
        "benchmark_source":     task["benchmark_source"],
        "benchmark_task_id":    task["task_id"],
        "turns":                turns,
        "turn_count":           len(turns),
        "final_agent_response": final_agent_response,
        "total_tokens_in":      total_tokens_in,
        "total_tokens_out":     total_tokens_out,
        "hit_turn_cap":         len(turns) == TURN_CAP and (not turns or turns[-1]["state_label"] not in TERMINAL_STATES),
        "failure_reason":       failure_reason,
    }


import uuid

test_session = run_one_session(
    condition="sff",
    persona_level="novice",
    task=SMOKE_TASK,
    session_id=f"smoke_session_{uuid.uuid4().hex[:8]}",
)

logger.info(f"Session: {test_session['session_id']}")
logger.info(f"Condition: {test_session['condition']}, persona: {test_session['persona_level']}, domain: {test_session['domain']}")
logger.info(f"Turn count: {test_session['turn_count']}")
logger.info(f"Hit turn cap: {test_session['hit_turn_cap']}")
logger.info(f"Failure reason: {test_session['failure_reason']}")
logger.info(f"Total tokens: in={test_session['total_tokens_in']}, out={test_session['total_tokens_out']}")
logger.info(f"\nState sequence:")
for t in test_session["turns"]:
    response_preview = t["visible_response"][:80].replace("\n", " ")
    logger.info(f"  turn {t['turn_index']} ({t['speaker']:7s}): {t['state_label']:25s} | {response_preview!r}")


In [ ]:
# Purpose: data processing / analysis step
logger.info("=" * 60)
logger.info("Diagnostic: raw persona outputs from the smoke session")
logger.info("=" * 60)
for t in test_session["turns"]:
    if t["speaker"] != "learner":
        continue
    logger.info(f"\n--- Turn {t['turn_index']} (learner) ---")
    logger.info(f"Parsed visible_response: {t['visible_response']!r}")
    logger.info(f"Parsed metadata:         {json.dumps(t['metadata'], indent=2)[:300]}")
    logger.info(f"Raw text (first 600 chars):")
    logger.info(t["raw_text"][:600])


In [ ]:
# Purpose: imports and small setup
import uuid

EXPERT_SMOKE_TASK = TASK_POOLS["python_debugging"][0]
EXPERT_SMOKE_SESSION_ID = f"smoke_expert_{uuid.uuid4().hex[:8]}"

logger.info("=" * 60)
logger.info(f"Smoke session (expert persona)")
logger.info(f"Task: {EXPERT_SMOKE_TASK['task_id']}  ({EXPERT_SMOKE_TASK['benchmark_source']}, {EXPERT_SMOKE_TASK['difficulty_tier']})")
logger.info(f"Session id: {EXPERT_SMOKE_SESSION_ID}")
logger.info("=" * 60)

expert_session = run_one_session(
    condition="sff",
    persona_level="expert",
    task=EXPERT_SMOKE_TASK,
    session_id=EXPERT_SMOKE_SESSION_ID,
)

logger.info(f"\nTurn count: {expert_session['turn_count']}")
logger.info(f"Hit turn cap: {expert_session['hit_turn_cap']}")
logger.info(f"Failure reason: {expert_session['failure_reason']}")
logger.info(f"Total tokens: in={expert_session['total_tokens_in']}, out={expert_session['total_tokens_out']}")

logger.info(f"\nState sequence and learner content:")
for t in expert_session["turns"]:
    response_preview = t["visible_response"][:120].replace("\n", " ")
    empty_marker = " <EMPTY>" if not t["visible_response"].strip() else ""
    logger.info(f"  turn {t['turn_index']} ({t['speaker']:7s}): {t['state_label']:25s}{empty_marker}")
    logger.info(f"    response: {response_preview!r}")


In [ ]:
# Purpose: defines function(s): call_openai_gpt4o
def call_openai_gpt4o(system: str, user: str, max_tokens: int = 1500) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = openai_client.chat.completions.create(
                model="gpt-4o",
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            )
            return {
                "text": r.choices[0].message.content,
                "tokens_in": r.usage.prompt_tokens,
                "tokens_out": r.usage.completion_tokens,
                "model": "gpt-4o",
                "stop_reason": r.choices[0].finish_reason,
            }
        except OPENAI_RETRYABLE as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_backoff(attempt))


def run_persona_turn_gpt4o(persona_level: str, task: dict, prior_turns: list, session_id: str) -> dict:
    prompt_name = PERSONA_PROMPT_BY_LEVEL[persona_level]
    variables = {
        "session_id":        session_id,
        "domain":             task["domain"],
        "difficulty_tier":    task["difficulty_tier"],
        "benchmark_source":   task["benchmark_source"],
        "benchmark_task_id":  task["task_id"],
        "prior_turns":        prior_turns,
    }
    rendered = render_prompt(prompt_name, variables)
    return call_with_json(
        call_openai_gpt4o,
        system=rendered,
        user="Produce the next learner turn following the system policy. Output JSON only.",
        max_tokens=1000,
        required_keys=PERSONA_REQUIRED_KEYS,
    )


GPT4O_SMOKE_TASK = TASK_POOLS["python_debugging"][0]
GPT4O_SESSION_ID = f"smoke_gpt4o_novice_{uuid.uuid4().hex[:8]}"

logger.info("=" * 60)
logger.info(f"Smoke session with GPT-4o for persona (novice)")
logger.info(f"Task: {GPT4O_SMOKE_TASK['task_id']}")
logger.info(f"Session id: {GPT4O_SESSION_ID}")
logger.info("=" * 60)

agent_r = run_agent_turn(condition="sff", task=GPT4O_SMOKE_TASK, persona_level="novice", session_id=GPT4O_SESSION_ID)
logger.info(f"\nAgent turn 0: {agent_r['parsed'].get('state_label')!r}")
logger.info(f"Agent response (first 200): {agent_r['parsed'].get('visible_response', '')[:200]!r}")

prior = [{"turn_index": 0, "speaker": "agent", "visible_response": agent_r['parsed']['visible_response']}]
persona_r = run_persona_turn_gpt4o(persona_level="novice", task=GPT4O_SMOKE_TASK, prior_turns=prior, session_id=GPT4O_SESSION_ID)
logger.info(f"\nPersona turn 1 (GPT-4o):")
logger.info(f"  state_label:    {persona_r['parsed'].get('state_label')!r}")
logger.info(f"  policy_action:  {persona_r['parsed'].get('policy_action')!r}")
logger.info(f"  visible_response: {persona_r['parsed'].get('visible_response', '')!r}")
logger.info(f"  tokens in/out:  {persona_r['tokens_in']} / {persona_r['tokens_out']}")


In [ ]:
# Purpose: data processing / analysis step
MODEL_PERSONA = "gpt-4o"


def call_openai(system: str, user: str, max_tokens: int = 1500) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = openai_client.chat.completions.create(
                model=MODEL_PERSONA,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            )
            return {
                "text": r.choices[0].message.content,
                "tokens_in": r.usage.prompt_tokens,
                "tokens_out": r.usage.completion_tokens,
                "model": MODEL_PERSONA,
                "stop_reason": r.choices[0].finish_reason,
            }
        except OPENAI_RETRYABLE as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_backoff(attempt))


smoke = call_openai(system="You are a test.", user="Reply with: ok")
logger.info(f"Persona model now: {smoke['model']}")
logger.info(f"Test call: in={smoke['tokens_in']} out={smoke['tokens_out']} text={smoke['text'].strip()!r}")


In [ ]:
# Purpose: defines function(s): call_openai_mini
def call_openai_mini(system: str, user: str, max_tokens: int = 1500) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            )
            return {
                "text": r.choices[0].message.content,
                "tokens_in": r.usage.prompt_tokens,
                "tokens_out": r.usage.completion_tokens,
                "model": "gpt-4o-mini",
                "stop_reason": r.choices[0].finish_reason,
            }
        except OPENAI_RETRYABLE as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_backoff(attempt))


def run_persona_turn_mini(persona_level: str, task: dict, prior_turns: list, session_id: str) -> dict:
    prompt_name = PERSONA_PROMPT_BY_LEVEL[persona_level]
    variables = {
        "session_id":        session_id,
        "domain":             task["domain"],
        "difficulty_tier":    task["difficulty_tier"],
        "benchmark_source":   task["benchmark_source"],
        "benchmark_task_id":  task["task_id"],
        "prior_turns":        prior_turns,
    }
    rendered = render_prompt(prompt_name, variables)
    return call_with_json(
        call_openai_mini,
        system=rendered,
        user="Produce the next learner turn following the system policy. Output JSON only.",
        max_tokens=1000,
        required_keys=PERSONA_REQUIRED_KEYS,
    )


MINI_SMOKE_TASK = TASK_POOLS["python_debugging"][0]
MINI_SESSION_ID = f"smoke_mini_novice_{uuid.uuid4().hex[:8]}"

logger.info("=" * 60)
logger.info(f"Smoke session with GPT-4o-mini for persona (novice)")
logger.info(f"Task: {MINI_SMOKE_TASK['task_id']}")
logger.info(f"Session id: {MINI_SESSION_ID}")
logger.info("=" * 60)

agent_r = run_agent_turn(condition="sff", task=MINI_SMOKE_TASK, persona_level="novice", session_id=MINI_SESSION_ID)
logger.info(f"\nAgent turn 0: {agent_r['parsed'].get('state_label')!r}")
logger.info(f"Agent response (first 200): {agent_r['parsed'].get('visible_response', '')[:200]!r}")

prior = [{"turn_index": 0, "speaker": "agent", "visible_response": agent_r['parsed']['visible_response']}]
persona_r = run_persona_turn_mini(persona_level="novice", task=MINI_SMOKE_TASK, prior_turns=prior, session_id=MINI_SESSION_ID)
logger.info(f"\nPersona turn 1 (GPT-4o-mini):")
logger.info(f"  state_label:      {persona_r['parsed'].get('state_label')!r}")
logger.info(f"  policy_action:    {persona_r['parsed'].get('policy_action')!r}")
logger.info(f"  visible_response: {persona_r['parsed'].get('visible_response', '')!r}")
logger.info(f"  tokens in/out:    {persona_r['tokens_in']} / {persona_r['tokens_out']}")


In [ ]:
# Purpose: data processing / analysis step
MODEL_PERSONA = "gpt-4o-mini"


def call_openai(system: str, user: str, max_tokens: int = 1500) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            r = openai_client.chat.completions.create(
                model=MODEL_PERSONA,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            )
            return {
                "text": r.choices[0].message.content,
                "tokens_in": r.usage.prompt_tokens,
                "tokens_out": r.usage.completion_tokens,
                "model": MODEL_PERSONA,
                "stop_reason": r.choices[0].finish_reason,
            }
        except OPENAI_RETRYABLE as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(_backoff(attempt))


smoke = call_openai(system="You are a test.", user="Reply with: ok")
logger.info(f"Persona model now locked to: {smoke['model']}")


In [ ]:
# Purpose: imports and small setup
import uuid
import random
import time

random.seed(20260605)

PILOT_PLAN = [
    ("seamless", "novice",       "python_debugging"),
    ("seamless", "intermediate", "python_debugging"),
    ("seamless", "expert",       "clinical_text_interpretation"),
    ("seamless", "novice",       "clinical_text_interpretation"),
    ("seamless", "intermediate", "historical_document_synthesis"),
    ("sff",      "novice",       "python_debugging"),
    ("sff",      "intermediate", "python_debugging"),
    ("sff",      "expert",       "clinical_text_interpretation"),
    ("sff",      "novice",       "clinical_text_interpretation"),
    ("sff",      "intermediate", "historical_document_synthesis"),
]

SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def pick_task(domain, used_ids):
    pool = TASK_POOLS[domain]
    candidates = [t for t in pool if t["task_id"] not in used_ids]
    task = random.choice(candidates if candidates else pool)
    used_ids.add(task["task_id"])
    return task


pilot_results = []
used_task_ids = set()
run_start = time.time()

for i, (condition, persona_level, domain) in enumerate(PILOT_PLAN):
    session_id = f"pilot_{i:02d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
    task = pick_task(domain, used_task_ids)

    logger.info(f"[{i+1:2d}/10] {condition:8s} | {persona_level:12s} | {domain:30s} | {task['benchmark_source']} {task['difficulty_tier']}")

    t0 = time.time()
    try:
        session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
    except Exception as e:
        logger.info(f"         SESSION CRASHED: {type(e).__name__}: {e}")
        append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
        pilot_results.append({"session_id": session_id, "status": "crashed", "error": str(e)})
        continue
    elapsed = time.time() - t0

    for turn in session["turns"]:
        append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

    tagger_record = None
    judge_record = None
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
        tagger_record = {"session_id": session_id, **tagger["parsed"]}
        append_jsonl(TAGGER_PATH, tagger_record)
    except Exception as e:
        logger.info(f"         tagger failed: {type(e).__name__}: {e}")

    try:
        judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
        judge_record = {"session_id": session_id, **judge["parsed"]}
        append_jsonl(JUDGE_PATH, judge_record)
    except Exception as e:
        logger.info(f"         judge failed: {type(e).__name__}: {e}")

    empty_learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner" and not t["visible_response"].strip())
    learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner")
    states = [t["state_label"] for t in session["turns"]]
    reached_terminal = states and states[-1] in TERMINAL_STATES

    pilot_results.append({
        "session_id": session_id,
        "status": "ok",
        "condition": condition,
        "persona_level": persona_level,
        "domain": domain,
        "turn_count": session["turn_count"],
        "hit_turn_cap": session["hit_turn_cap"],
        "reached_terminal": reached_terminal,
        "empty_learner_turns": empty_learner_turns,
        "learner_turns": learner_turns,
        "states": states,
        "tokens_in": session["total_tokens_in"],
        "tokens_out": session["total_tokens_out"],
        "judge_score": judge_record.get("domain_raw_score") if judge_record else None,
        "elapsed_sec": round(elapsed, 1),
    })
    logger.info(f"         turns={session['turn_count']} terminal={reached_terminal} empty_learner={empty_learner_turns}/{learner_turns} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
logger.info(f"\nPilot run complete: {len([r for r in pilot_results if r['status']=='ok'])}/10 ok, {run_elapsed:.0f}s total")


In [ ]:
# Purpose: imports and small setup
import inspect
for fn in [run_agent_turn, run_persona_turn, run_judge, run_state_tagger]:
    src = inspect.getsource(fn)
    assert "render_prompt" in src, f"{fn.__name__} does not use render_prompt"
    assert '"""' not in src.split("render_prompt")[0][-200:], "possible inline prompt"
logger.info("All role functions render from the loaded PROMPTS dict only. No inline prompts.")


In [ ]:
# Purpose: data processing / analysis step
# (sanitized path) logger.info("State sequences per sessioPROJECT_ROOT")
for r in pilot_results:
    if r["status"] != "ok":
        logger.info(f"{r['session_id']}: CRASHED")
        continue
    seq = " -> ".join(s if s else "None" for s in r["states"])
    logger.info(f"{r['condition']:8s} {r['persona_level']:12s} {r['domain'][:12]:12s}")
    logger.info(f"   {seq}")
    logger.info(f"   judge_score: {r['judge_score']}")
    logger.info()

logger.info("=" * 60)
logger.info("Distinct state labels observed across all sessions:")
all_states = set()
for r in pilot_results:
    if r["status"] == "ok":
        all_states.update(s for s in r["states"] if s)
for s in sorted(all_states):
    terminal_marker = "  <-- in TERMINAL_STATES" if s in TERMINAL_STATES else ""
    logger.info(f"   {s!r}{terminal_marker}")

logger.info("=" * 60)
logger.info(f"TERMINAL_STATES set: {TERMINAL_STATES}")


In [ ]:
# Purpose: imports and small setup
import re

# 1. Show the exact state_label schema line from each prompt that emits one
logger.info("=" * 60)
logger.info("How state_label is written in the prompt schemas")
logger.info("=" * 60)
for name in ["persona_novice", "sff_agent", "seamless_agent", "state_tagger"]:
    text = PROMPTS[name]
    for line in text.splitlines():
        if "state_label" in line or ("state" in line.lower() and "S2" in line):
            logger.info(f"\n[{name}]")
            logger.info(f"  {line.strip()[:200]}")

# 2. Show raw model output for a few learner + agent turns from the pilot logs
logger.info("\n" + "=" * 60)
logger.info("Raw state_label values actually emitted (from sessions.jsonl)")
logger.info("=" * 60)
import json
with open(SESSIONS_PATH) as f:
    rows = [json.loads(l) for l in f if l.strip()]

from collections import Counter
by_speaker = {"agent": Counter(), "learner": Counter()}
for r in rows:
    by_speaker[r["speaker"]][r.get("state_label")] += 1

for speaker in ["agent", "learner"]:
    logger.info(f"\n{speaker} state_label values:")
    for label, count in by_speaker[speaker].most_common():
        is_clean = label in {
            "S0_START","S1_TASK_PRESENTED","S2_INITIAL_ORIENTATION","S3_DIRECT_ANSWER",
            "S4_SOCRATIC_PROMPT","S5_USER_REVISION","S6_USER_ACCEPT","S7_USER_REJECT",
            "S8_FEEDBACK_LOOP","S9_TASK_COMPLETE","S10_FAILURE"
        }
        marker = "" if is_clean else "  <-- INVALID"
        logger.info(f"   {count:2d}x  {label!r}{marker}")


In [ ]:
# Purpose: data processing / analysis step
# Write corrected prompts directly into Drive prompts/ folder
backup_dir = PROMPTS_DIR / "_backup_pre_pilot"
backup_dir.mkdir(exist_ok=True)

corrected = {}

corrected["persona_novice.md"] = PROMPTS_DIR.joinpath("persona_novice.md").read_text(encoding="utf-8") \
    .replace(
        '"state_label": "S2_INITIAL_ORIENTATION | S5_USER_REVISION | S6_USER_ACCEPT | S7_USER_REJECT | S8_FEEDBACK_LOOP",',
        '"state_label": "<choose exactly one: S2_INITIAL_ORIENTATION, S5_USER_REVISION, S6_USER_ACCEPT, S7_USER_REJECT, S8_FEEDBACK_LOOP>",'
    ).replace(
        '"policy_action": "learner_attempt | learner_revision | learner_acceptance | learner_challenge | clarifying_question | passive_acceptance",',
        '"policy_action": "<choose exactly one: learner_attempt, learner_revision, learner_acceptance, learner_challenge, clarifying_question, passive_acceptance>",'
    )

corrected["persona_intermediate.md"] = PROMPTS_DIR.joinpath("persona_intermediate.md").read_text(encoding="utf-8") \
    .replace(
        '"state_label": "S2_INITIAL_ORIENTATION | S5_USER_REVISION | S6_USER_ACCEPT | S7_USER_REJECT | S8_FEEDBACK_LOOP",',
        '"state_label": "<choose exactly one: S2_INITIAL_ORIENTATION, S5_USER_REVISION, S6_USER_ACCEPT, S7_USER_REJECT, S8_FEEDBACK_LOOP>",'
    ).replace(
        '"policy_action": "learner_attempt | learner_revision | learner_acceptance | learner_challenge | clarifying_question | critical_appraisal",',
        '"policy_action": "<choose exactly one: learner_attempt, learner_revision, learner_acceptance, learner_challenge, clarifying_question, critical_appraisal>",'
    )

corrected["persona_expert.md"] = PROMPTS_DIR.joinpath("persona_expert.md").read_text(encoding="utf-8") \
    .replace(
        '"state_label": "S2_INITIAL_ORIENTATION | S5_USER_REVISION | S6_USER_ACCEPT | S7_USER_REJECT | S8_FEEDBACK_LOOP",',
        '"state_label": "<choose exactly one: S2_INITIAL_ORIENTATION, S5_USER_REVISION, S6_USER_ACCEPT, S7_USER_REJECT, S8_FEEDBACK_LOOP>",'
    ).replace(
        '"policy_action": "learner_attempt | learner_revision | learner_acceptance | learner_challenge | clarifying_question | evidence_verification",',
        '"policy_action": "<choose exactly one: learner_attempt, learner_revision, learner_acceptance, learner_challenge, clarifying_question, evidence_verification>",'
    )

corrected["sff_treatment_agent.md"] = PROMPTS_DIR.joinpath("sff_treatment_agent.md").read_text(encoding="utf-8") \
    .replace(
        '"state_label": "S2_INITIAL_ORIENTATION | S4_SOCRATIC_PROMPT | S8_FEEDBACK_LOOP | S9_TASK_COMPLETE",',
        '"state_label": "<choose exactly one: S2_INITIAL_ORIENTATION, S4_SOCRATIC_PROMPT, S8_FEEDBACK_LOOP, S9_TASK_COMPLETE>",'
    ).replace(
        '"policy_action": "orientation_question | attention_cue | contrast_cue | misconception_cue | partial_correction | completion_support | reflection_prompt",',
        '"policy_action": "<choose exactly one: orientation_question, attention_cue, contrast_cue, misconception_cue, partial_correction, completion_support, reflection_prompt>",'
    )

for fname, new_text in corrected.items():
    src = PROMPTS_DIR / fname
    original = src.read_text(encoding="utf-8")
    # back up original
    (backup_dir / fname).write_text(original, encoding="utf-8")
    # confirm the replacement actually changed something
    if "<choose exactly one:" not in new_text:
        logger.info(f"  {fname}: WARNING — replacement did not apply, file left unchanged")
        continue
    src.write_text(new_text, encoding="utf-8")
    changed = sum(1 for a, b in zip(original.splitlines(), new_text.splitlines()) if a != b)
    logger.info(f"  {fname}: written, {changed} lines changed")

logger.info("\nDone. Originals backed up to prompts/_backup_pre_pilot/")


In [ ]:
# Purpose: data processing / analysis step
# Re-read prompts from Drive
for name, filename in PROMPT_FILES.items():
    path = PROMPTS_DIR / filename
    if not path.exists():
        logger.info(f"MISSING: {filename}")
        continue
    PROMPTS[name] = path.read_text(encoding="utf-8")

# (sanitized path) logger.info("Checking the four edited files for the fiPROJECT_ROOT")
for name in ["persona_novice", "persona_intermediate", "persona_expert", "sff_agent"]:
    text = PROMPTS[name]
    has_fix = "<choose exactly one:" in text
    has_old = "ORIENTATION | S" in text
    status = "FIXED" if (has_fix and not has_old) else ("STILL OLD" if has_old else "UNCLEAR")
    logger.info(f"  {name:24s} {status}")
    for line in text.splitlines():
        if '"state_label"' in line:
            logger.info(f"      {line.strip()[:90]}")

logger.info("\nAll files currently in prompts folder:")
for p in sorted(PROMPTS_DIR.glob("*")):
    logger.info(f"  {p.name}")


In [ ]:
# Purpose: data processing / analysis step
gdoc = PROMPTS_DIR / "sff_treatment_agent.md.gdoc"
if gdoc.exists():
    gdoc.unlink()
    logger.info("Deleted the stray .gdoc")
else:
    logger.info("Already gone")
logger.info("Prompt folder now:")
for p in sorted(PROMPTS_DIR.glob("*.md")):
    logger.info(f"  {p.name}")


In [ ]:
# Purpose: imports and small setup
import inspect
logger.info(inspect.getsource(run_one_session))


In [ ]:
# Purpose: defines function(s): run_one_session
def run_one_session(condition: str, persona_level: str, task: dict, session_id: str) -> dict:
    turns = []
    prior_turns_view = []
    total_tokens_in = 0
    total_tokens_out = 0
    failure_reason = None

    for turn_index in range(TURN_CAP):
        try:
            if turn_index % 2 == 0:
                role = "agent"
                result = run_agent_turn(condition=condition, task=task, persona_level=persona_level, session_id=session_id)
            else:
                role = "learner"
                result = run_persona_turn(persona_level=persona_level, task=task, prior_turns=prior_turns_view, session_id=session_id)
        except Exception as e:
            failure_reason = f"{role}_call_failed_turn_{turn_index}: {type(e).__name__}: {e}"
            break

        parsed = result["parsed"]
        raw_state = parsed.get("state_label")
        clean_state, state_valid = normalize_state_label(raw_state)
        turn_record = {
            "turn_index":       turn_index,
            "speaker":          role,
            "state_label":      raw_state,
            "state_label_valid": state_valid,
            "policy_action":    parsed.get("policy_action"),
            "visible_response": parsed.get("visible_response", ""),
            "metadata":         parsed.get("metadata", {}),
            "model":            result["model"],
            "tokens_in":        result["tokens_in"],
            "tokens_out":       result["tokens_out"],
            "attempts":         result["attempts"],
            "raw_text":         result["raw_text"],
        }
        turns.append(turn_record)
        prior_turns_view.append({
            "turn_index":       turn_index,
            "speaker":          role,
            "visible_response": parsed.get("visible_response", ""),
        })
        total_tokens_in += result["tokens_in"]
        total_tokens_out += result["tokens_out"]

        if state_valid and clean_state in TERMINAL_STATES:
            break

    final_agent_response = next(
        (t["visible_response"] for t in reversed(turns) if t["speaker"] == "agent"),
        "",
    )

    invalid_label_count = sum(1 for t in turns if not t["state_label_valid"])

    return {
        "session_id":           session_id,
        "condition":            condition,
        "persona_level":        persona_level,
        "domain":               task["domain"],
        "difficulty_tier":      task["difficulty_tier"],
        "benchmark_source":     task["benchmark_source"],
        "benchmark_task_id":    task["task_id"],
        "turns":                turns,
        "turn_count":           len(turns),
        "final_agent_response": final_agent_response,
        "total_tokens_in":      total_tokens_in,
        "total_tokens_out":     total_tokens_out,
        "invalid_label_count":  invalid_label_count,
        "hit_turn_cap":         len(turns) == TURN_CAP and (not turns or not (turns[-1]["state_label_valid"] and normalize_state_label(turns[-1]["state_label"])[0] in TERMINAL_STATES)),
        "failure_reason":       failure_reason,
    }


In [ ]:
# Purpose: imports and small setup
import inspect
src = inspect.getsource(run_one_session)
logger.info("state_label_valid" in src and "invalid_label_count" in src)
logger.info("normalize_state_label" in src)


In [ ]:
# Purpose: data processing / analysis step
VALID_STATES = {
    "S0_START", "S1_TASK_PRESENTED", "S2_INITIAL_ORIENTATION", "S3_DIRECT_ANSWER",
    "S4_SOCRATIC_PROMPT", "S5_USER_REVISION", "S6_USER_ACCEPT", "S7_USER_REJECT",
    "S8_FEEDBACK_LOOP", "S9_TASK_COMPLETE", "S10_FAILURE",
}

def normalize_state_label(raw_label):
    """Returns (clean_label_or_None, is_valid). Flags pipe-menus and out-of-vocab labels."""
    if raw_label in VALID_STATES:
        return raw_label, True
    return None, False

logger.info("VALID_STATES defined:", len(VALID_STATES), "states")
logger.info(normalize_state_label("S9_TASK_COMPLETE"))   # expect ('S9_TASK_COMPLETE', True)
logger.info(normalize_state_label("S2 | S5 | S6"))        # expect (None, False)
logger.info(normalize_state_label("S3_ACTIVE_PROCESSING"))# expect (None, False)


In [ ]:
# Purpose: load run/session data from disk
RUN_ID = "pilot_002"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
logger.info(f"Logging to: {RUN_DIR}")


In [ ]:
# Purpose: imports and small setup
import uuid
import random
import time

random.seed(20260605)

PILOT_PLAN = [
    ("seamless", "novice",       "python_debugging"),
    ("seamless", "intermediate", "python_debugging"),
    ("seamless", "expert",       "clinical_text_interpretation"),
    ("seamless", "novice",       "clinical_text_interpretation"),
    ("seamless", "intermediate", "historical_document_synthesis"),
    ("sff",      "novice",       "python_debugging"),
    ("sff",      "intermediate", "python_debugging"),
    ("sff",      "expert",       "clinical_text_interpretation"),
    ("sff",      "novice",       "clinical_text_interpretation"),
    ("sff",      "intermediate", "historical_document_synthesis"),
]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def pick_task(domain, used_ids):
    pool = TASK_POOLS[domain]
    candidates = [t for t in pool if t["task_id"] not in used_ids]
    task = random.choice(candidates if candidates else pool)
    used_ids.add(task["task_id"])
    return task


pilot_results = []
used_task_ids = set()
run_start = time.time()

for i, (condition, persona_level, domain) in enumerate(PILOT_PLAN):
    session_id = f"pilot_{i:02d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
    task = pick_task(domain, used_task_ids)

    logger.info(f"[{i+1:2d}/10] {condition:8s} | {persona_level:12s} | {domain:30s} | {task['benchmark_source']} {task['difficulty_tier']}")

    t0 = time.time()
    try:
        session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
    except Exception as e:
        logger.info(f"         SESSION CRASHED: {type(e).__name__}: {e}")
        append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
        pilot_results.append({"session_id": session_id, "status": "crashed", "error": str(e)})
        continue
    elapsed = time.time() - t0

    for turn in session["turns"]:
        append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

    tagger_record = None
    judge_record = None
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
        tagger_record = {"session_id": session_id, **tagger["parsed"]}
        append_jsonl(TAGGER_PATH, tagger_record)
    except Exception as e:
        logger.info(f"         tagger failed: {type(e).__name__}: {e}")

    try:
        judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
        judge_record = {"session_id": session_id, **judge["parsed"]}
        append_jsonl(JUDGE_PATH, judge_record)
    except Exception as e:
        logger.info(f"         judge failed: {type(e).__name__}: {e}")

    empty_learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner" and not t["visible_response"].strip())
    learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner")
    states = [t["state_label"] for t in session["turns"]]
    reached_terminal = any(normalize_state_label(s)[0] in TERMINAL_STATES for s in states)

    pilot_results.append({
        "session_id": session_id,
        "status": "ok",
        "condition": condition,
        "persona_level": persona_level,
        "domain": domain,
        "turn_count": session["turn_count"],
        "hit_turn_cap": session["hit_turn_cap"],
        "reached_terminal": reached_terminal,
        "empty_learner_turns": empty_learner_turns,
        "learner_turns": learner_turns,
        "invalid_label_count": session["invalid_label_count"],
        "states": states,
        "tokens_in": session["total_tokens_in"],
        "tokens_out": session["total_tokens_out"],
        "judge_score": judge_record.get("domain_raw_score") if judge_record else None,
        "elapsed_sec": round(elapsed, 1),
    })
    logger.info(f"         turns={session['turn_count']} terminal={reached_terminal} empty_learner={empty_learner_turns}/{learner_turns} invalid_labels={session['invalid_label_count']} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
logger.info(f"\nPilot run complete: {len([r for r in pilot_results if r['status']=='ok'])}/10 ok, {run_elapsed:.0f}s total")


In [ ]:
# Purpose: data processing / analysis step
logger.info("State sequences per session (pilot_002):\n")
for r in pilot_results:
    if r["status"] != "ok":
        logger.info(f"{r['session_id']}: CRASHED")
        continue
    seq = " -> ".join(s if s else "None" for s in r["states"])
    logger.info(f"{r['condition']:8s} {r['persona_level']:12s} {r['domain'][:12]:12s}  invalid={r['invalid_label_count']}")
    logger.info(f"   {seq}")
    logger.info()

logger.info("=" * 60)
logger.info("Distinct state labels now observed:")
all_states = set()
for r in pilot_results:
    if r["status"] == "ok":
        all_states.update(s for s in r["states"] if s)
for s in sorted(all_states):
    valid = "" if s in VALID_STATES else "  <-- INVALID"
    logger.info(f"   {s!r}{valid}")


In [ ]:
# Purpose: defines function(s): print_transcript
def print_transcript(session_id_prefix):
    rows = []
    with open(SESSIONS_PATH) as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                if r["session_id"].startswith(session_id_prefix):
                    rows.append(r)
    if not rows:
        logger.info(f"No turns found for {session_id_prefix}")
        return
    logger.info(f"\n{'='*70}\nSESSION: {rows[0]['session_id']}\n{'='*70}")
    for r in sorted(rows, key=lambda x: x["turn_index"]):
        logger.info(f"\n--- turn {r['turn_index']} [{r['speaker']}] state={r['state_label']} action={r.get('policy_action')}")
        logger.info(r["visible_response"][:500])

# one SFF python session (the all-S2 one) and one seamless python session
logger.info("Transcript: " + "pilot_05_sff_novice")
logger.info("Transcript: " + "pilot_00_seamless_novice")


In [ ]:
# Purpose: imports and small setup
import inspect
logger.info("=== run_agent_turn source ===")
logger.info(inspect.getsource(run_agent_turn))
logger.info("\n=== does sff_agent prompt have a prior_turns slot? ===")
for slot in ["prior_turns", "conversation", "history", "transcript", "prior"]:
    logger.info(f"  {{{{{slot}}}}}: {'YES' if ('{{'+slot+'}}') in PROMPTS['sff_agent'] else 'no'}")
logger.info("\n=== all {{...}} slots in sff_agent prompt ===")
import re
logger.info(sorted(set(re.findall(r'\{\{([a-z_]+)\}\}', PROMPTS['sff_agent']))))
logger.info("\n=== all {{...}} slots in seamless_agent prompt ===")
logger.info(sorted(set(re.findall(r'\{\{([a-z_]+)\}\}', PROMPTS['seamless_agent']))))


In [ ]:
# Purpose: data processing / analysis step
logger.info(PROMPTS_DIR, len(TASK_POOLS), "run_one_session" in dir())


In [ ]:
# Purpose: data processing / analysis step
backup_dir = PROMPTS_DIR / "_backup_pre_history_fix"
backup_dir.mkdir(exist_ok=True)

for fname in ["sff_treatment_agent.md", "seamless_control_agent.md"]:
    path = PROMPTS_DIR / fname
    text = path.read_text(encoding="utf-8")
    (backup_dir / fname).write_text(text, encoding="utf-8")

# (sanitized path)     old_ctx = "ContexPROJECT_ROOT\n"
# (sanitized path)     new_ctx = "ContexPROJECT_ROOT\n\nConversation so far (prior turns):\n{{prior_turns}}\n"
    if "{{prior_turns}}" not in text:
        assert text.count(old_ctx) == 1, f"{fname}: context block not found uniquely"
        text = text.replace(old_ctx, new_ctx)

    if fname == "seamless_control_agent.md":
        old_pa = '"policy_action": "direct_explanation | direct_correction | final_answer | clarification_request | edge_case_note",'
        new_pa = '"policy_action": "<choose exactly one: direct_explanation, direct_correction, final_answer, clarification_request, edge_case_note>",'
        text = text.replace(old_pa, new_pa)

    path.write_text(text, encoding="utf-8")
    logger.info(f"{fname}: prior_turns slot = {'{{prior_turns}}' in text}")

logger.info("Done. Backups in prompts/_backup_pre_history_fix/")


In [ ]:
# Purpose: imports and small setup
import re
for name in ["sff_agent", "seamless_agent"]:
    fname = PROMPT_FILES[name]
    text = (PROMPTS_DIR / fname).read_text(encoding="utf-8")
    PROMPTS[name] = text
    slots = sorted(set(re.findall(r'\{\{([a-z_]+)\}\}', text)))
    logger.info(f"{name}: prior_turns={'prior_turns' in slots}")


In [ ]:
# Purpose: defines function(s): run_agent_turn
def run_agent_turn(condition: str, task: dict, persona_level: str, session_id: str, prior_turns: list = None, task_context: str = "") -> dict:
    prompt_name = "sff_agent" if condition == "sff" else "seamless_agent"
    variables = {
        "session_id":        session_id,
        "domain":            task["domain"],
        "persona_level":     persona_level,
        "difficulty_tier":   task["difficulty_tier"],
        "benchmark_source":  task["benchmark_source"],
        "benchmark_task_id": task["task_id"],
        "task_prompt":       task["task_prompt"],
        "task_context":      task_context or task.get("context", "") or "",
        "prior_turns":       prior_turns if prior_turns is not None else [],
    }
    rendered = render_prompt(prompt_name, variables)
    return call_with_json(
        call_anthropic,
        system=rendered,
        user="Produce the next agent turn following the system policy. Output JSON only.",
        max_tokens=1500,
        required_keys=AGENT_REQUIRED_KEYS,
    )


In [ ]:
# Purpose: imports and small setup
import inspect
logger.info("prior_turns" in inspect.signature(run_agent_turn).parameters)


In [ ]:
# Purpose: imports and small setup
import inspect
logger.info(inspect.getsource(run_one_session))


In [ ]:
# Purpose: defines function(s): run_one_session
def run_one_session(condition: str, persona_level: str, task: dict, session_id: str) -> dict:
    turns = []
    prior_turns_view = []
    total_tokens_in = 0
    total_tokens_out = 0
    failure_reason = None

    for turn_index in range(TURN_CAP):
        try:
            if turn_index % 2 == 0:
                role = "agent"
                result = run_agent_turn(condition=condition, task=task, persona_level=persona_level, session_id=session_id, prior_turns=prior_turns_view)
            else:
                role = "learner"
                result = run_persona_turn(persona_level=persona_level, task=task, prior_turns=prior_turns_view, session_id=session_id)
        except Exception as e:
            failure_reason = f"{role}_call_failed_turn_{turn_index}: {type(e).__name__}: {e}"
            break

        parsed = result["parsed"]
        raw_state = parsed.get("state_label")
        clean_state, state_valid = normalize_state_label(raw_state)
        turn_record = {
            "turn_index":       turn_index,
            "speaker":          role,
            "state_label":      raw_state,
            "state_label_valid": state_valid,
            "policy_action":    parsed.get("policy_action"),
            "visible_response": parsed.get("visible_response", ""),
            "metadata":         parsed.get("metadata", {}),
            "model":            result["model"],
            "tokens_in":        result["tokens_in"],
            "tokens_out":       result["tokens_out"],
            "attempts":         result["attempts"],
            "raw_text":         result["raw_text"],
        }
        turns.append(turn_record)
        prior_turns_view.append({
            "turn_index":       turn_index,
            "speaker":          role,
            "visible_response": parsed.get("visible_response", ""),
        })
        total_tokens_in += result["tokens_in"]
        total_tokens_out += result["tokens_out"]

        if state_valid and clean_state in TERMINAL_STATES:
            break

    final_agent_response = next(
        (t["visible_response"] for t in reversed(turns) if t["speaker"] == "agent"),
        "",
    )

    invalid_label_count = sum(1 for t in turns if not t["state_label_valid"])

    return {
        "session_id":           session_id,
        "condition":            condition,
        "persona_level":        persona_level,
        "domain":               task["domain"],
        "difficulty_tier":      task["difficulty_tier"],
        "benchmark_source":     task["benchmark_source"],
        "benchmark_task_id":    task["task_id"],
        "turns":                turns,
        "turn_count":           len(turns),
        "final_agent_response": final_agent_response,
        "total_tokens_in":      total_tokens_in,
        "total_tokens_out":     total_tokens_out,
        "invalid_label_count":  invalid_label_count,
        "hit_turn_cap":         len(turns) == TURN_CAP and (not turns or not (turns[-1]["state_label_valid"] and normalize_state_label(turns[-1]["state_label"])[0] in TERMINAL_STATES)),
        "failure_reason":       failure_reason,
    }


In [ ]:
# Purpose: imports and small setup
import inspect
src = inspect.getsource(run_one_session)
logger.info("agent gets history:", "session_id=session_id, prior_turns=prior_turns_view)" in src)


In [ ]:
# Purpose: load run/session data from disk
RUN_ID = "pilot_003"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
logger.info(f"Logging to: {RUN_DIR}")


In [ ]:
# Purpose: imports and small setup
import uuid
import random
import time

random.seed(20260605)

PILOT_PLAN = [
    ("seamless", "novice",       "python_debugging"),
    ("seamless", "intermediate", "python_debugging"),
    ("seamless", "expert",       "clinical_text_interpretation"),
    ("seamless", "novice",       "clinical_text_interpretation"),
    ("seamless", "intermediate", "historical_document_synthesis"),
    ("sff",      "novice",       "python_debugging"),
    ("sff",      "intermediate", "python_debugging"),
    ("sff",      "expert",       "clinical_text_interpretation"),
    ("sff",      "novice",       "clinical_text_interpretation"),
    ("sff",      "intermediate", "historical_document_synthesis"),
]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def pick_task(domain, used_ids):
    pool = TASK_POOLS[domain]
    candidates = [t for t in pool if t["task_id"] not in used_ids]
    task = random.choice(candidates if candidates else pool)
    used_ids.add(task["task_id"])
    return task


pilot_results = []
used_task_ids = set()
run_start = time.time()

for i, (condition, persona_level, domain) in enumerate(PILOT_PLAN):
    session_id = f"pilot_{i:02d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
    task = pick_task(domain, used_task_ids)

    logger.info(f"[{i+1:2d}/10] {condition:8s} | {persona_level:12s} | {domain:30s} | {task['benchmark_source']} {task['difficulty_tier']}")

    t0 = time.time()
    try:
        session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
    except Exception as e:
        logger.info(f"         SESSION CRASHED: {type(e).__name__}: {e}")
        append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
        pilot_results.append({"session_id": session_id, "status": "crashed", "error": str(e)})
        continue
    elapsed = time.time() - t0

    for turn in session["turns"]:
        append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

    tagger_record = None
    judge_record = None
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
        tagger_record = {"session_id": session_id, **tagger["parsed"]}
        append_jsonl(TAGGER_PATH, tagger_record)
    except Exception as e:
        logger.info(f"         tagger failed: {type(e).__name__}: {e}")

    try:
        judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
        judge_record = {"session_id": session_id, **judge["parsed"]}
        append_jsonl(JUDGE_PATH, judge_record)
    except Exception as e:
        logger.info(f"         judge failed: {type(e).__name__}: {e}")

    empty_learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner" and not t["visible_response"].strip())
    learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner")
    states = [t["state_label"] for t in session["turns"]]
    reached_terminal = any(normalize_state_label(s)[0] in TERMINAL_STATES for s in states)

    pilot_results.append({
        "session_id": session_id,
        "status": "ok",
        "condition": condition,
        "persona_level": persona_level,
        "domain": domain,
        "turn_count": session["turn_count"],
        "hit_turn_cap": session["hit_turn_cap"],
        "reached_terminal": reached_terminal,
        "empty_learner_turns": empty_learner_turns,
        "learner_turns": learner_turns,
        "invalid_label_count": session["invalid_label_count"],
        "states": states,
        "tokens_in": session["total_tokens_in"],
        "tokens_out": session["total_tokens_out"],
        "judge_score": judge_record.get("domain_raw_score") if judge_record else None,
        "elapsed_sec": round(elapsed, 1),
    })
    logger.info(f"         turns={session['turn_count']} terminal={reached_terminal} empty_learner={empty_learner_turns}/{learner_turns} invalid_labels={session['invalid_label_count']} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
logger.info(f"\nPilot run complete: {len([r for r in pilot_results if r['status']=='ok'])}/10 ok, {run_elapsed:.0f}s total")


In [ ]:
# Purpose: data processing / analysis step
logger.info("SFF state sequences from pilot_003:\n")
for r in pilot_results:
    if r["status"] != "ok" or r["condition"] != "sff":
        continue
    seq = " -> ".join(s if s else "None" for s in r["states"])
    logger.info(f"sff {r['persona_level']:12s} {r['domain'][:12]:12s}")
    logger.info(f"   {seq}\n")

# pull two full SFF transcripts to see if the agent is escalating or spinning
def print_transcript(prefix):
    rows = []
    with open(SESSIONS_PATH) as f:
        for line in f:
            if line.strip():
                rr = json.loads(line)
                if rr["session_id"].startswith(prefix):
                    rows.append(rr)
    if not rows:
        logger.info(f"(no turns for {prefix})"); return
    logger.info(f"\n{'='*68}\n{rows[0]['session_id']}\n{'='*68}")
    for r in sorted(rows, key=lambda x: x["turn_index"]):
        logger.info(f"\n--- t{r['turn_index']} [{r['speaker']}] {r['state_label']} / {r.get('policy_action')}")
        logger.info(r["visible_response"][:450])

logger.info("Transcript: " + "pilot_05_sff_novice")
logger.info("Transcript: " + "pilot_06_sff_intermediate")


In [ ]:
# Purpose: imports and small setup
from google.colab import userdata
import os
# (redacted) os.environ["GROQ_API_KEY"] = userdata.get("Groq")
logger.info("Groq key loaded, length:", len(userdata.get("Groq")))


In [ ]:
# Purpose: data processing / analysis step
try:
    test = call_groq(
        system="You are a test. Reply with exactly: OK",
        user="Say OK",
        max_tokens=10,
    )
    logger.info("GROQ WORKS:", test.get("raw_text", test))
except Exception as e:
    logger.info("GROQ STILL FAILING:", type(e).__name__, str(e)[:300])


In [ ]:
# Purpose: load run/session data from disk
RUN_ID = "pilot_003b"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
logger.info(f"Logging to: {RUN_DIR}")


In [ ]:
# Purpose: imports and small setup
import uuid
import random
import time

random.seed(20260605)

PILOT_PLAN = [
    ("seamless", "novice",       "python_debugging"),
    ("seamless", "intermediate", "python_debugging"),
    ("seamless", "expert",       "clinical_text_interpretation"),
    ("seamless", "novice",       "clinical_text_interpretation"),
    ("seamless", "intermediate", "historical_document_synthesis"),
    ("sff",      "novice",       "python_debugging"),
    ("sff",      "intermediate", "python_debugging"),
    ("sff",      "expert",       "clinical_text_interpretation"),
    ("sff",      "novice",       "clinical_text_interpretation"),
    ("sff",      "intermediate", "historical_document_synthesis"),
]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def pick_task(domain, used_ids):
    pool = TASK_POOLS[domain]
    candidates = [t for t in pool if t["task_id"] not in used_ids]
    task = random.choice(candidates if candidates else pool)
    used_ids.add(task["task_id"])
    return task


pilot_results = []
used_task_ids = set()
run_start = time.time()

for i, (condition, persona_level, domain) in enumerate(PILOT_PLAN):
    session_id = f"pilot_{i:02d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
    task = pick_task(domain, used_task_ids)

    logger.info(f"[{i+1:2d}/10] {condition:8s} | {persona_level:12s} | {domain:30s} | {task['benchmark_source']} {task['difficulty_tier']}")

    t0 = time.time()
    try:
        session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
    except Exception as e:
        logger.info(f"         SESSION CRASHED: {type(e).__name__}: {e}")
        append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
        pilot_results.append({"session_id": session_id, "status": "crashed", "error": str(e)})
        continue
    elapsed = time.time() - t0

    for turn in session["turns"]:
        append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

    tagger_record = None
    judge_record = None
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
        tagger_record = {"session_id": session_id, **tagger["parsed"]}
        append_jsonl(TAGGER_PATH, tagger_record)
    except Exception as e:
        logger.info(f"         tagger failed: {type(e).__name__}: {e}")

    try:
        judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
        judge_record = {"session_id": session_id, **judge["parsed"]}
        append_jsonl(JUDGE_PATH, judge_record)
    except Exception as e:
        logger.info(f"         judge failed: {type(e).__name__}: {e}")

    empty_learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner" and not t["visible_response"].strip())
    learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner")
    states = [t["state_label"] for t in session["turns"]]
    reached_terminal = any(normalize_state_label(s)[0] in TERMINAL_STATES for s in states)

    pilot_results.append({
        "session_id": session_id,
        "status": "ok",
        "condition": condition,
        "persona_level": persona_level,
        "domain": domain,
        "turn_count": session["turn_count"],
        "hit_turn_cap": session["hit_turn_cap"],
        "reached_terminal": reached_terminal,
        "empty_learner_turns": empty_learner_turns,
        "learner_turns": learner_turns,
        "invalid_label_count": session["invalid_label_count"],
        "states": states,
        "tokens_in": session["total_tokens_in"],
        "tokens_out": session["total_tokens_out"],
        "judge_score": judge_record.get("domain_raw_score") if judge_record else None,
        "elapsed_sec": round(elapsed, 1),
    })
    logger.info(f"         turns={session['turn_count']} terminal={reached_terminal} empty_learner={empty_learner_turns}/{learner_turns} invalid_labels={session['invalid_label_count']} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
logger.info(f"\nPilot run complete: {len([r for r in pilot_results if r['status']=='ok'])}/10 ok, {run_elapsed:.0f}s total")


In [ ]:
# Purpose: data processing / analysis step
try:
    big = call_groq(system="You are a test.", user="Write a 200-word paragraph about clouds.", max_tokens=400)
    logger.info("BIG CALL OK — cap is lifted:", big.get("tokens_out"), "tokens out")
except Exception as e:
    logger.info("STILL CAPPED:", str(e)[:200])


In [ ]:
# Purpose: imports and small setup
import json
with open(SESSIONS_PATH) as f:
    first = json.loads(f.readline())
logger.info("Keys stored per turn:")
for k in first.keys():
    val = str(first[k])[:80]
    logger.info(f"  {k}: {val}")

# how many sessions are logged
with open(SESSIONS_PATH) as f:
    sids = set(json.loads(l)["session_id"] for l in f if l.strip())
logger.info(f"\nLogged sessions in pilot_003b: {len(sids)}")


In [ ]:
# Purpose: load run/session data from disk
RUN_ID = "pilot_004"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
TASKS_PATH = RUN_DIR / "session_tasks.jsonl"   # NEW: records which task each session used
logger.info(f"Logging to: {RUN_DIR}")
logger.info(f"Fresh folder, empty: {not any(RUN_DIR.glob('*.jsonl'))}")


In [ ]:
# Purpose: imports and small setup
import uuid
import random
import time

random.seed(20260605)

PILOT_PLAN = [
    ("seamless", "novice",       "python_debugging"),
    ("seamless", "intermediate", "python_debugging"),
    ("seamless", "expert",       "clinical_text_interpretation"),
    ("seamless", "novice",       "clinical_text_interpretation"),
    ("seamless", "intermediate", "historical_document_synthesis"),
    ("sff",      "novice",       "python_debugging"),
    ("sff",      "intermediate", "python_debugging"),
    ("sff",      "expert",       "clinical_text_interpretation"),
    ("sff",      "novice",       "clinical_text_interpretation"),
    ("sff",      "intermediate", "historical_document_synthesis"),
]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def pick_task(domain, used_ids):
    pool = TASK_POOLS[domain]
    candidates = [t for t in pool if t["task_id"] not in used_ids]
    task = random.choice(candidates if candidates else pool)
    used_ids.add(task["task_id"])
    return task


pilot_results = []
used_task_ids = set()
run_start = time.time()

for i, (condition, persona_level, domain) in enumerate(PILOT_PLAN):
    session_id = f"pilot_{i:02d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
    task = pick_task(domain, used_task_ids)

    # NEW: record which task this session used (for reproducibility + re-judging from disk)
    append_jsonl(TASKS_PATH, {"session_id": session_id, **task})

    logger.info(f"[{i+1:2d}/10] {condition:8s} | {persona_level:12s} | {domain:30s} | {task['benchmark_source']} {task['difficulty_tier']}")

    t0 = time.time()
    try:
        session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
    except Exception as e:
        logger.info(f"         SESSION CRASHED: {type(e).__name__}: {e}")
        append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
        pilot_results.append({"session_id": session_id, "status": "crashed", "error": str(e)})
        continue
    elapsed = time.time() - t0

    for turn in session["turns"]:
        append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

    tagger_record = None
    judge_record = None
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
        tagger_record = {"session_id": session_id, **tagger["parsed"]}
        append_jsonl(TAGGER_PATH, tagger_record)
    except Exception as e:
        logger.info(f"         tagger failed: {type(e).__name__}: {e}")

    try:
        judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
        judge_record = {"session_id": session_id, **judge["parsed"]}
        append_jsonl(JUDGE_PATH, judge_record)
    except Exception as e:
        logger.info(f"         judge failed: {type(e).__name__}: {e}")

    empty_learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner" and not t["visible_response"].strip())
    learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner")
    states = [t["state_label"] for t in session["turns"]]
    reached_terminal = any(normalize_state_label(s)[0] in TERMINAL_STATES for s in states)

    pilot_results.append({
        "session_id": session_id,
        "status": "ok",
        "condition": condition,
        "persona_level": persona_level,
        "domain": domain,
        "turn_count": session["turn_count"],
        "hit_turn_cap": session["hit_turn_cap"],
        "reached_terminal": reached_terminal,
        "empty_learner_turns": empty_learner_turns,
        "learner_turns": learner_turns,
        "invalid_label_count": session["invalid_label_count"],
        "states": states,
        "tokens_in": session["total_tokens_in"],
        "tokens_out": session["total_tokens_out"],
        "judge_score": judge_record.get("score_value") if judge_record else None,
        "tagged": tagger_record is not None,
        "elapsed_sec": round(elapsed, 1),
    })
    logger.info(f"         turns={session['turn_count']} terminal={reached_terminal} empty_learner={empty_learner_turns}/{learner_turns} invalid_labels={session['invalid_label_count']} judge={'OK' if judge_record else 'FAIL'} tag={'OK' if tagger_record else 'FAIL'} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
ok = len([r for r in pilot_results if r['status']=='ok'])
judged = len([r for r in pilot_results if r.get('judge_score') is not None])
tagged = len([r for r in pilot_results if r.get('tagged')])
logger.info(f"\nPilot complete: {ok}/10 ok, {judged}/10 judged, {tagged}/10 tagged, {run_elapsed:.0f}s")


In [ ]:
# Purpose: imports and small setup
from google.colab import userdata
import os
# (redacted) os.environ["GROQ_API_KEY"] = userdata.get("Groq")
logger.info("loaded, length:", len(userdata.get("Groq")))


In [ ]:
# Purpose: data processing / analysis step
for i in range(3):
    try:
        r = call_groq(system="test", user="Write 150 words about rivers.", max_tokens=400)
        logger.info(i, "OK", r.get("tokens_out"))
    except Exception as e:
        logger.info(i, "FAIL", str(e)[:120])


In [ ]:
# Purpose: data processing / analysis step
logger.info(RUN_DIR)
logger.info("empty:", not any(RUN_DIR.glob('*.jsonl')))


In [ ]:
# Purpose: load run/session data from disk
RUN_ID = "pilot_005"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
TASKS_PATH = RUN_DIR / "session_tasks.jsonl"
logger.info(f"Logging to: {RUN_DIR}, empty: {not any(RUN_DIR.glob('*.jsonl'))}")


In [ ]:
# Purpose: data processing / analysis step
GROQ_KEY_TEST = "REDACTED_GROQ_KEY"

import os
# (redacted) os.environ["GROQ_API_KEY"] = GROQ_KEY_TEST
logger.info("Key set, length:", len(GROQ_KEY_TEST))

# now test it directly
try:
    r = call_groq(system="test", user="Reply with one sentence about mountains.", max_tokens=100)
    logger.info("GROQ OK:", r.get("tokens_out"), "tokens")
except Exception as e:
    logger.info("FAIL:", str(e)[:250])


In [ ]:
# Purpose: data processing / analysis step
for i in range(4):
    try:
        r = call_groq(system="test", user="Write 200 words about the ocean.", max_tokens=400)
        logger.info(i, "OK", r.get("tokens_out"))
    except Exception as e:
        logger.info(i, "FAIL", str(e)[:120])


In [ ]:
# Purpose: imports and small setup
import inspect
logger.info(inspect.getsource(run_judge))
logger.info("="*40)
logger.info(inspect.getsource(run_state_tagger))
logger.info("="*40)
logger.info(inspect.getsource(call_anthropic))


In [ ]:
# Purpose: execution-based scoring of code solutions
# TEMPORARY: run judge + tagger on Claude for a PLUMBING TEST ONLY.
# This is NOT valid study data — Claude grading Claude breaks the model-family
# separation the design requires. Final data must be judged on Groq (Llama 3.3 70B).
JUDGE_BACKEND = "claude"   # set back to "groq" once Groq billing is fixed

def _judge_caller():
    return call_anthropic if JUDGE_BACKEND == "claude" else call_groq

def run_judge(task: dict, agent_response: str, session_id: str, test_pass_rate=None, local_execution_logs: str = "") -> dict:
    variables = {
        "session_id":             session_id,
        "domain":                 task["domain"],
        "difficulty_tier":        task["difficulty_tier"],
        "benchmark_source":       task["benchmark_source"],
        "benchmark_task_id":      task["task_id"],
        "task_prompt":            task["task_prompt"],
        "ground_truth":           task["ground_truth"],
        "frozen_rubric":          task.get("rubric", ""),
        "required_output_schema": task.get("task_type", ""),
        "test_pass_rate":         test_pass_rate if test_pass_rate is not None else "not_applicable",
        "local_execution_logs":   local_execution_logs or "not_applicable",
        "agent_response":         agent_response,
    }
    rendered = render_prompt("judge", variables)
    return call_with_json(
        _judge_caller(),
        system=rendered,
        user="Audit the final session and produce the blinded score. Output JSON only.",
        max_tokens=2000,
        required_keys=JUDGE_REQUIRED_KEYS,
    )

def run_state_tagger(task: dict, prior_turns: list, session_id: str) -> dict:
    variables = {
        "session_id":      session_id,
        "domain":          task["domain"],
        "difficulty_tier": task["difficulty_tier"],
        "prior_turns":     prior_turns,
    }
    rendered = render_prompt("state_tagger", variables)
    return call_with_json(
        _judge_caller(),
        system=rendered,
        user="Re-tag the session blind. Output JSON only.",
        max_tokens=2000,
        required_keys=STATE_TAGGER_REQUIRED_KEYS,
    )

logger.info(f"Judge/tagger backend: {JUDGE_BACKEND}")


In [ ]:
# Purpose: load run/session data from disk
RUN_ID = "pilot_006_CLAUDE_JUDGE_PLUMBING_TEST"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
TASKS_PATH = RUN_DIR / "session_tasks.jsonl"
logger.info(f"Logging to: {RUN_DIR}, empty: {not any(RUN_DIR.glob('*.jsonl'))}")


In [ ]:
# Purpose: imports and small setup
import uuid
import random
import time

random.seed(20260605)

PILOT_PLAN = [
    ("seamless", "novice",       "python_debugging"),
    ("seamless", "intermediate", "python_debugging"),
    ("seamless", "expert",       "clinical_text_interpretation"),
    ("seamless", "novice",       "clinical_text_interpretation"),
    ("seamless", "intermediate", "historical_document_synthesis"),
    ("sff",      "novice",       "python_debugging"),
    ("sff",      "intermediate", "python_debugging"),
    ("sff",      "expert",       "clinical_text_interpretation"),
    ("sff",      "novice",       "clinical_text_interpretation"),
    ("sff",      "intermediate", "historical_document_synthesis"),
]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def pick_task(domain, used_ids):
    pool = TASK_POOLS[domain]
    candidates = [t for t in pool if t["task_id"] not in used_ids]
    task = random.choice(candidates if candidates else pool)
    used_ids.add(task["task_id"])
    return task


pilot_results = []
used_task_ids = set()
run_start = time.time()

for i, (condition, persona_level, domain) in enumerate(PILOT_PLAN):
    session_id = f"pilot_{i:02d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
    task = pick_task(domain, used_task_ids)

    append_jsonl(TASKS_PATH, {"session_id": session_id, **task})

    logger.info(f"[{i+1:2d}/10] {condition:8s} | {persona_level:12s} | {domain:30s} | {task['benchmark_source']} {task['difficulty_tier']}")

    t0 = time.time()
    try:
        session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
    except Exception as e:
        logger.info(f"         SESSION CRASHED: {type(e).__name__}: {e}")
        append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
        pilot_results.append({"session_id": session_id, "status": "crashed", "error": str(e)})
        continue
    elapsed = time.time() - t0

    for turn in session["turns"]:
        append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

    tagger_record = None
    judge_record = None
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
        tagger_record = {"session_id": session_id, **tagger["parsed"]}
        append_jsonl(TAGGER_PATH, tagger_record)
    except Exception as e:
        logger.info(f"         tagger failed: {type(e).__name__}: {e}")

    try:
        judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
        judge_record = {"session_id": session_id, **judge["parsed"]}
        append_jsonl(JUDGE_PATH, judge_record)
    except Exception as e:
        logger.info(f"         judge failed: {type(e).__name__}: {e}")

    empty_learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner" and not t["visible_response"].strip())
    learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner")
    states = [t["state_label"] for t in session["turns"]]
    reached_terminal = any(normalize_state_label(s)[0] in TERMINAL_STATES for s in states)

    pilot_results.append({
        "session_id": session_id,
        "status": "ok",
        "condition": condition,
        "persona_level": persona_level,
        "domain": domain,
        "turn_count": session["turn_count"],
        "hit_turn_cap": session["hit_turn_cap"],
        "reached_terminal": reached_terminal,
        "empty_learner_turns": empty_learner_turns,
        "learner_turns": learner_turns,
        "invalid_label_count": session["invalid_label_count"],
        "states": states,
        "tokens_in": session["total_tokens_in"],
        "tokens_out": session["total_tokens_out"],
        "judge_score": judge_record.get("score_value") if judge_record else None,
        "tagged": tagger_record is not None,
        "elapsed_sec": round(elapsed, 1),
    })
    logger.info(f"         turns={session['turn_count']} terminal={reached_terminal} empty_learner={empty_learner_turns}/{learner_turns} invalid_labels={session['invalid_label_count']} judge={'OK' if judge_record else 'FAIL'} tag={'OK' if tagger_record else 'FAIL'} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
ok = len([r for r in pilot_results if r['status']=='ok'])
judged = len([r for r in pilot_results if r.get('judge_score') is not None])
tagged = len([r for r in pilot_results if r.get('tagged')])
logger.info(f"\nPilot complete: {ok}/10 ok, {judged}/10 judged, {tagged}/10 tagged, {run_elapsed:.0f}s")


In [ ]:
# Purpose: imports and small setup
import json

# inline labels: last state per session from sessions.jsonl
inline = {}
with open(SESSIONS_PATH) as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            inline.setdefault(r["session_id"], []).append((r["turn_index"], r["state_label"]))

# what the judge actually returns (so we fix the score field)
logger.info("=== one judge record (to find the real score field) ===")
with open(JUDGE_PATH) as f:
    jr = json.loads(f.readline())
for k, v in jr.items():
    logger.info(f"  {k}: {str(v)[:70]}")

# what the tagger returns
logger.info("\n=== one tagger record (to see its label format) ===")
with open(TAGGER_PATH) as f:
    tr = json.loads(f.readline())
for k, v in tr.items():
    logger.info(f"  {k}: {str(v)[:70]}")


In [ ]:
# Purpose: imports and small setup
import json

# fix the judge tally with the REAL field
judge_scores = {}
with open(JUDGE_PATH) as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            drs = r.get("domain_raw_score", {})
            judge_scores[r["session_id"]] = drs.get("score_value")

logger.info("Judge scores (real field domain_raw_score.score_value):")
for sid, sc in judge_scores.items():
    logger.info(f"  {sid[:45]:45s} {sc}/4")
logger.info(f"  -> {sum(1 for v in judge_scores.values() if v is not None)}/10 actually scored\n")

# inline final states
inline_final = {}
with open(SESSIONS_PATH) as f:
    rows = [json.loads(l) for l in f if l.strip()]
for sid in set(r["session_id"] for r in rows):
    turns = sorted([r for r in rows if r["session_id"]==sid], key=lambda x: x["turn_index"])
    inline_final[sid] = turns[-1]["state_label"]

# blinded tagger terminal states + per-turn agreement
logger.info("Inline vs blinded tagger comparison:")
total_turns = 0
agree_turns = 0
with open(TAGGER_PATH) as f:
    for line in f:
        if not line.strip(): continue
        tr = json.loads(line)
        sid = tr["session_id"]
        tagger_seq = tr.get("state_sequence", [])
        inline_turns = sorted([r for r in rows if r["session_id"]==sid], key=lambda x: x["turn_index"])
        # align by turn_index
        tagger_by_idx = {t["turn_index"]: t.get("state") for t in tagger_seq}
        matches = 0
        for it in inline_turns:
            ti = it["turn_index"]
            if ti in tagger_by_idx:
                total_turns += 1
                if tagger_by_idx[ti] == it["state_label"]:
                    matches += 1; agree_turns += 1
        logger.info(f"  {sid[:42]:42s} inline_final={inline_final[sid]:22s} tagger_terminal={tr.get('terminal_state')}  turn_agree={matches}/{len(inline_turns)}")

logger.info(f"\nOverall per-turn label agreement (inline vs blinded): {agree_turns}/{total_turns} = {agree_turns/total_turns*100:.0f}%")


In [ ]:
# Purpose: imports and small setup
import json

rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
tagger = {json.loads(l)["session_id"]: json.loads(l) for l in open(TAGGER_PATH) if l.strip()}

# focus on the SFF sessions where agreement was weak
for sid in sorted(set(r["session_id"] for r in rows)):
    if "sff" not in sid:
        continue
    inline_turns = sorted([r for r in rows if r["session_id"]==sid], key=lambda x: x["turn_index"])
    tseq = {t["turn_index"]: t.get("state") for t in tagger.get(sid, {}).get("state_sequence", [])}
    logger.info(f"\n{'='*70}\n{sid}")
    logger.info(f"{'turn':4s} {'speaker':8s} {'INLINE':24s} {'BLINDED TAGGER':24s} match")
    for it in inline_turns:
        ti = it["turn_index"]
        inl = it["state_label"]
        tag = tseq.get(ti, "—")
        mark = "" if inl == tag else "  <-- DIFF"
        logger.info(f"{ti:<4d} {it['speaker']:8s} {inl:24s} {str(tag):24s}{mark}")


In [ ]:
# Purpose: data processing / analysis step
FRIEND_KEY = "REDACTED_GROQ_KEY"

import os
# (redacted) os.environ["GROQ_API_KEY"] = FRIEND_KEY
logger.info("Key set, length:", len(FRIEND_KEY))


In [ ]:
# Purpose: data processing / analysis step
ok = 0
for i in range(4):
    try:
        big_input = "Analyze this transcript and respond in detail. " + ("turn data " * 400)
        r = call_groq(system="You are an evaluator.", user=big_input, max_tokens=2000)
        logger.info(i, "OK", r.get("tokens_out"))
        ok += 1
    except Exception as e:
        cap = "Limit 100000" in str(e)
        # extract org id from the error to confirm which account this key hits
        org = ""
        if "organization" in str(e):
            org = str(e).split("organization")[1][:35]
        logger.info(i, "FAIL", "DAILY-CAP" if cap else "other", "org:"+org)
logger.info(f"\n{ok}/4 succeeded")


In [ ]:
# Purpose: imports and small setup
import inspect
logger.info(inspect.getsource(call_groq))
logger.info("="*40)
# find where the groq client is created
import re
for name in dir():
    if 'groq' in name.lower() and 'client' in name.lower():
        logger.info("client var:", name)


In [ ]:
# Purpose: imports and small setup
from groq import Groq

# rebuild the client with the friend's key (the one you set in FRIEND_KEY)
groq_client = Groq(api_key=FRIEND_KEY)
logger.info("Groq client rebuilt with new key")


In [ ]:
# Purpose: data processing / analysis step
ok = 0
for i in range(4):
    try:
        big_input = "Analyze this transcript and respond in detail. " + ("turn data " * 400)
        r = call_groq(system="You are an evaluator.", user=big_input, max_tokens=2000)
        logger.info(i, "OK", r.get("tokens_out"))
        ok += 1
    except Exception as e:
        cap = "Limit 100000" in str(e)
        org = str(e).split("organization")[1][:35] if "organization" in str(e) else ""
        logger.info(i, "FAIL", "DAILY-CAP" if cap else "other", "org:"+org)
logger.info(f"\n{ok}/4 succeeded")


In [ ]:
# Purpose: data processing / analysis step
JUDGE_BACKEND = "groq"
logger.info("backend:", JUDGE_BACKEND)


In [ ]:
# Purpose: load run/session data from disk
RUN_ID = "pilot_007_GROQ_REAL"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
TASKS_PATH = RUN_DIR / "session_tasks.jsonl"
logger.info(f"Logging to: {RUN_DIR}, empty: {not any(RUN_DIR.glob('*.jsonl'))}")


In [ ]:
# Purpose: imports and small setup
import uuid
import random
import time

random.seed(20260605)

PILOT_PLAN = [
    ("seamless", "novice",       "python_debugging"),
    ("seamless", "intermediate", "python_debugging"),
    ("seamless", "expert",       "clinical_text_interpretation"),
    ("seamless", "novice",       "clinical_text_interpretation"),
    ("seamless", "intermediate", "historical_document_synthesis"),
    ("sff",      "novice",       "python_debugging"),
    ("sff",      "intermediate", "python_debugging"),
    ("sff",      "expert",       "clinical_text_interpretation"),
    ("sff",      "novice",       "clinical_text_interpretation"),
    ("sff",      "intermediate", "historical_document_synthesis"),
]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def pick_task(domain, used_ids):
    pool = TASK_POOLS[domain]
    candidates = [t for t in pool if t["task_id"] not in used_ids]
    task = random.choice(candidates if candidates else pool)
    used_ids.add(task["task_id"])
    return task


pilot_results = []
used_task_ids = set()
run_start = time.time()

for i, (condition, persona_level, domain) in enumerate(PILOT_PLAN):
    session_id = f"pilot_{i:02d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
    task = pick_task(domain, used_task_ids)

    append_jsonl(TASKS_PATH, {"session_id": session_id, **task})

    logger.info(f"[{i+1:2d}/10] {condition:8s} | {persona_level:12s} | {domain:30s} | {task['benchmark_source']} {task['difficulty_tier']}")

    t0 = time.time()
    try:
        session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
    except Exception as e:
        logger.info(f"         SESSION CRASHED: {type(e).__name__}: {e}")
        append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
        pilot_results.append({"session_id": session_id, "status": "crashed", "error": str(e)})
        continue
    elapsed = time.time() - t0

    for turn in session["turns"]:
        append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

    tagger_record = None
    judge_record = None
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
        tagger_record = {"session_id": session_id, **tagger["parsed"]}
        append_jsonl(TAGGER_PATH, tagger_record)
    except Exception as e:
        logger.info(f"         tagger failed: {type(e).__name__}: {e}")

    try:
        judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
        judge_record = {"session_id": session_id, **judge["parsed"]}
        append_jsonl(JUDGE_PATH, judge_record)
    except Exception as e:
        logger.info(f"         judge failed: {type(e).__name__}: {e}")

    empty_learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner" and not t["visible_response"].strip())
    learner_turns = sum(1 for t in session["turns"] if t["speaker"] == "learner")
    states = [t["state_label"] for t in session["turns"]]
    reached_terminal = any(normalize_state_label(s)[0] in TERMINAL_STATES for s in states)

    pilot_results.append({
        "session_id": session_id,
        "status": "ok",
        "condition": condition,
        "persona_level": persona_level,
        "domain": domain,
        "turn_count": session["turn_count"],
        "hit_turn_cap": session["hit_turn_cap"],
        "reached_terminal": reached_terminal,
        "empty_learner_turns": empty_learner_turns,
        "learner_turns": learner_turns,
        "invalid_label_count": session["invalid_label_count"],
        "states": states,
        "tokens_in": session["total_tokens_in"],
        "tokens_out": session["total_tokens_out"],
        "judge_score": (judge_record.get("domain_raw_score", {}) or {}).get("score_value") if judge_record else None,
        "tagged": tagger_record is not None,
        "elapsed_sec": round(elapsed, 1),
    })
    logger.info(f"         turns={session['turn_count']} terminal={reached_terminal} empty_learner={empty_learner_turns}/{learner_turns} invalid_labels={session['invalid_label_count']} judge={'OK' if judge_record else 'FAIL'} tag={'OK' if tagger_record else 'FAIL'} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
ok = len([r for r in pilot_results if r['status']=='ok'])
judged = len([r for r in pilot_results if r.get('judge_score') is not None])
tagged = len([r for r in pilot_results if r.get('tagged')])
logger.info(f"\nPilot complete: {ok}/10 ok, {judged}/10 judged, {tagged}/10 tagged, {run_elapsed:.0f}s")


In [ ]:
# Purpose: imports and small setup
import json

rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
tagger = {json.loads(l)["session_id"]: json.loads(l) for l in open(TAGGER_PATH) if l.strip()}
judge = {json.loads(l)["session_id"]: json.loads(l) for l in open(JUDGE_PATH) if l.strip()}

# judge score spread
logger.info("Judge scores (real Groq):")
for sid in sorted(judge):
    sc = (judge[sid].get("domain_raw_score") or {}).get("score_value")
    cond = "sff" if "sff" in sid else "seam"
    logger.info(f"  {cond:4s} {sid[:40]:40s} {sc}/4")

# inline vs blinded tagger agreement
total = agree = 0
for sid in set(r["session_id"] for r in rows):
    inline_turns = sorted([r for r in rows if r["session_id"]==sid], key=lambda x: x["turn_index"])
    tseq = {t["turn_index"]: t.get("state") for t in tagger.get(sid, {}).get("state_sequence", [])}
    for it in inline_turns:
        if it["turn_index"] in tseq:
            total += 1
            if tseq[it["turn_index"]] == it["state_label"]:
                agree += 1

logger.info(f"\nInline vs blinded tagger agreement (real Groq): {agree}/{total} = {agree/total*100:.0f}%")


In [ ]:
# Purpose: data processing / analysis step
backup_dir = PROMPTS_DIR / "_backup_pre_tagger_fix"
backup_dir.mkdir(exist_ok=True)

path = PROMPTS_DIR / "state_tagger.md"
text = path.read_text(encoding="utf-8")
(backup_dir / "state_tagger.md").write_text(text, encoding="utf-8")  # backup

# 1. add rule 8 constraining terminal_state
old_rule = '7. Check whether the sequence reaches a valid terminal state and whether process evidence needed for downstream Research Question 4 analysis is present.\n'
new_rule = old_rule + '8. The "terminal_state" field MUST be exactly one of: "S9_TASK_COMPLETE" (session reached valid completion), "S10_FAILURE" (session failed or exceeded turn cap), or "NONE_REACHED" (session ended without a terminal state, e.g. hit the turn budget mid-process). Do not use any other value, free text, or labels like "INCOMPLETE" or "none".\n'

# 2. annotate the schema field
old_schema = '  "terminal_state": "",'
new_schema = '  "terminal_state": "<exactly one of: S9_TASK_COMPLETE, S10_FAILURE, NONE_REACHED>",'

if "NONE_REACHED" in text:
    logger.info("Already patched, skipping.")
else:
    assert old_rule in text, "rule 7 anchor not found"
    assert old_schema in text, "terminal_state schema line not found"
    text = text.replace(old_rule, new_rule).replace(old_schema, new_schema)
    path.write_text(text, encoding="utf-8")
    logger.info("state_tagger.md patched in Drive. Backup in _backup_pre_tagger_fix/")

# reload into memory
PROMPTS["state_tagger"] = path.read_text(encoding="utf-8")
logger.info("terminal_state rule present:", "NONE_REACHED" in PROMPTS["state_tagger"])


In [ ]:
# Purpose: imports and small setup
import json

# load existing pilot_007 transcripts + their tasks
rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
tasks = {json.loads(l)["session_id"]: json.loads(l) for l in open(TASKS_PATH) if l.strip()}

sessions = {}
for r in rows:
    sessions.setdefault(r["session_id"], []).append(r)

VALID_TERMINAL = {"S9_TASK_COMPLETE", "S10_FAILURE", "NONE_REACHED"}
results = []
for sid, turns in sessions.items():
    turns = sorted(turns, key=lambda x: x["turn_index"])
    prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in turns]
    task = tasks.get(sid)
    if task is None:
        logger.info(f"{sid[:40]}: no task record, skip")
        continue
    try:
        tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=sid)
        ts = tagger["parsed"].get("terminal_state")
        valid = ts in VALID_TERMINAL
        # also check every per-turn state is in vocabulary
        seq = tagger["parsed"].get("state_sequence", [])
        bad_states = [s.get("state") for s in seq if s.get("state") not in VALID_STATES]
        results.append((sid, ts, valid, len(bad_states)))
        logger.info(f"{sid[:40]:40s} terminal={ts:18s} valid={valid} bad_turn_states={len(bad_states)}")
    except Exception as e:
        logger.info(f"{sid[:40]:40s} FAILED: {str(e)[:60]}")

clean = sum(1 for _,_,v,b in results if v and b==0)
logger.info(f"\n{clean}/{len(results)} sessions fully clean (valid terminal + all turn states in vocab)")


In [ ]:
# Purpose: imports and small setup
import inspect
checks = {
    "run_agent_turn has prior_turns": "prior_turns" in inspect.signature(run_agent_turn).parameters,
    "run_one_session passes history to agent": "prior_turns=prior_turns_view)" in inspect.getsource(run_one_session),
    "normalize_state_label exists": "normalize_state_label" in dir(),
    "VALID_STATES exists": "VALID_STATES" in dir(),
    "groq_client is rebuilt": "groq_client" in dir(),
    "JUDGE_BACKEND is groq": globals().get("JUDGE_BACKEND") == "groq",
}
for k, v in checks.items():
    logger.info(f"  {'OK ' if v else 'MISSING'} {k}")


In [ ]:
# Purpose: imports and small setup
import inspect

logger.info("# ===== CONSTANTS =====")
for name in ["PROJECT_ROOT", "PROMPTS_DIR", "TURN_CAP", "MAX_RETRIES", "MODEL_HAIKU", "MODEL_OPENAI", "MODEL_LLAMA", "MODEL_PERSONA"]:
    if name in dir():
        logger.info(f"{name} = {repr(globals()[name])}")
logger.info("TERMINAL_STATES =", TERMINAL_STATES)
logger.info("VALID_STATES =", VALID_STATES)
logger.info("PROMPT_FILES =", PROMPT_FILES)
logger.info("AGENT_REQUIRED_KEYS =", globals().get("AGENT_REQUIRED_KEYS"))
logger.info("PERSONA_REQUIRED_KEYS =", globals().get("PERSONA_REQUIRED_KEYS"))
logger.info("JUDGE_REQUIRED_KEYS =", globals().get("JUDGE_REQUIRED_KEYS"))
logger.info("STATE_TAGGER_REQUIRED_KEYS =", globals().get("STATE_TAGGER_REQUIRED_KEYS"))


In [ ]:
# Purpose: data processing / analysis step
ok = 0
for i in range(3):
    try:
        big_input = "Analyze this transcript. " + ("turn data " * 400)
        r = call_groq(system="You are an evaluator.", user=big_input, max_tokens=2000)
        logger.info(i, "OK", r.get("tokens_out"))
        ok += 1
    except Exception as e:
        cap = "Limit 100000" in str(e)
        org = str(e).split("organization")[1][:35] if "organization" in str(e) else ""
        logger.info(i, "FAIL", "DAILY-CAP" if cap else "other", "org:"+org)
logger.info(f"{ok}/3")


In [ ]:
# Purpose: imports and small setup
import uuid, random, time
from collections import defaultdict

random.seed(20260606)

# fresh batch folder
RUN_ID = "batch_001"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
TASKS_PATH = RUN_DIR / "session_tasks.jsonl"

CONDITIONS = ["seamless", "sff"]
DOMAINS = ["python_debugging", "clinical_text_interpretation", "historical_document_synthesis"]
PERSONAS = ["novice", "intermediate", "expert"]
SESSIONS_PER_CELL = 3   # 18 cells x 3 = 54

# difficulty-aware task picker: rotate across tiers within a domain
def pick_task_stratified(domain, used_ids, tier_cycle):
    pool = TASK_POOLS[domain]
    tier = next(tier_cycle)
    tier_tasks = [t for t in pool if t["difficulty_tier"] == tier and t["task_id"] not in used_ids]
    if not tier_tasks:  # fall back to any unused, then any
        tier_tasks = [t for t in pool if t["task_id"] not in used_ids] or pool
    task = random.choice(tier_tasks)
    used_ids.add(task["task_id"])
    return task

def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# verify pools have difficulty tiers
import itertools
for d in DOMAINS:
    tiers = set(t["difficulty_tier"] for t in TASK_POOLS[d])
    logger.info(f"{d}: {len(TASK_POOLS[d])} tasks, tiers={sorted(tiers)}")

total = len(CONDITIONS)*len(DOMAINS)*len(PERSONAS)*SESSIONS_PER_CELL
logger.info(f"\nWill generate {total} sessions ({SESSIONS_PER_CELL} per cell x 18 cells)")
logger.info(f"Logging to: {RUN_DIR}, empty: {not any(RUN_DIR.glob('*.jsonl'))}")


In [ ]:
# Purpose: data processing / analysis step
batch_results = []
used_task_ids = set()
run_start = time.time()
n = 0

# tier cycle per domain so each cell rotates easy/medium/hard
tier_cycles = {d: itertools.cycle(["easy", "medium", "hard"]) for d in DOMAINS}

for condition in CONDITIONS:
    for domain in DOMAINS:
        for persona_level in PERSONAS:
            for s in range(SESSIONS_PER_CELL):
                n += 1
                session_id = f"b1_{n:03d}_{condition}_{persona_level}_{uuid.uuid4().hex[:6]}"
                task = pick_task_stratified(domain, used_task_ids, tier_cycles[domain])
                append_jsonl(TASKS_PATH, {"session_id": session_id, **task})

                t0 = time.time()
                try:
                    session = run_one_session(condition=condition, persona_level=persona_level, task=task, session_id=session_id)
                except Exception as e:
                    logger.info(f"[{n:2d}/54] CRASH {condition}/{domain[:8]}/{persona_level}: {type(e).__name__}: {str(e)[:60]}")
                    append_jsonl(INVALID_PATH, {"session_id": session_id, "reason": f"crash: {type(e).__name__}: {e}"})
                    batch_results.append({"session_id": session_id, "status": "crashed"})
                    continue
                elapsed = time.time() - t0

                for turn in session["turns"]:
                    append_jsonl(SESSIONS_PATH, {"session_id": session_id, **turn})

                tagger_record = judge_record = None
                prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
                try:
                    tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=session_id)
                    tagger_record = {"session_id": session_id, **tagger["parsed"]}
                    append_jsonl(TAGGER_PATH, tagger_record)
                except Exception as e:
                    logger.info(f"        tagger fail: {str(e)[:60]}")
                try:
                    judge = run_judge(task=task, agent_response=session["final_agent_response"], session_id=session_id)
                    judge_record = {"session_id": session_id, **judge["parsed"]}
                    append_jsonl(JUDGE_PATH, judge_record)
                except Exception as e:
                    logger.info(f"        judge fail: {str(e)[:60]}")

                states = [t["state_label"] for t in session["turns"]]
                reached_terminal = any(normalize_state_label(s)[0] in TERMINAL_STATES for s in states)
                jscore = (judge_record.get("domain_raw_score", {}) or {}).get("score_value") if judge_record else None

                batch_results.append({
                    "session_id": session_id, "status": "ok", "condition": condition,
                    "persona_level": persona_level, "domain": domain,
                    "difficulty_tier": task["difficulty_tier"], "turn_count": session["turn_count"],
                    "reached_terminal": reached_terminal, "invalid_label_count": session["invalid_label_count"],
                    "judge_score": jscore, "tagged": tagger_record is not None,
                })

                if n % 6 == 0 or n <= 3:
                    done = len([r for r in batch_results if r["status"]=="ok"])
                    logger.info(f"[{n:2d}/54] {condition:8s} {domain[:10]:10s} {persona_level:11s} {task['difficulty_tier']:6s} term={reached_terminal} inv={session['invalid_label_count']} j={'OK' if judge_record else 'X'} t={'OK' if tagger_record else 'X'} {elapsed:.0f}s")

run_elapsed = time.time() - run_start
ok = len([r for r in batch_results if r["status"]=="ok"])
judged = len([r for r in batch_results if r.get("judge_score") is not None])
tagged = len([r for r in batch_results if r.get("tagged")])
crashed = len([r for r in batch_results if r["status"]=="crashed"])
logger.info(f"\nBatch complete: {ok}/54 ok, {judged}/54 judged, {tagged}/54 tagged, {crashed} crashed, {run_elapsed/60:.1f} min")


In [ ]:
# Purpose: imports and small setup
import json
from collections import defaultdict

bj = [json.loads(l) for l in open(JUDGE_PATH) if l.strip()]
bt = {json.loads(l)["session_id"]: json.loads(l) for l in open(TAGGER_PATH) if l.strip()}
br = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]

# map session -> condition
cond = {}
for r in batch_results:
    if r["status"]=="ok":
        cond[r["session_id"]] = r["condition"]

# 1. judge score distribution by condition
logger.info("=== Judge scores by condition ===")
scores = defaultdict(list)
for j in bj:
    sid = j["session_id"]
    sc = (j.get("domain_raw_score") or {}).get("score_value")
    if sc is not None and sid in cond:
        scores[cond[sid]].append(sc)
for c in ["seamless","sff"]:
    s = scores[c]
    logger.info(f"  {c:8s} n={len(s)} mean={sum(s)/len(s):.2f} dist={ {v:s.count(v) for v in sorted(set(s))} }")

# 2. terminal state by condition (from blinded tagger)
logger.info("\n=== Blinded tagger terminal_state by condition ===")
term = defaultdict(lambda: defaultdict(int))
for sid, t in bt.items():
    if sid in cond:
        term[cond[sid]][t.get("terminal_state")] += 1
for c in ["seamless","sff"]:
    logger.info(f"  {c:8s} {dict(term[c])}")

# 3. trajectory length by condition (turns per session)
logger.info("\n=== Mean turns per session by condition ===")
turns_by = defaultdict(list)
for r in batch_results:
    if r["status"]=="ok":
        turns_by[r["condition"]].append(r["turn_count"])
for c in ["seamless","sff"]:
    t = turns_by[c]
    logger.info(f"  {c:8s} mean={sum(t)/len(t):.1f} turns")

# 4. state diversity by condition (distinct states per session, from tagger)
logger.info("\n=== Mean distinct states per session (blinded tagger) ===")
div = defaultdict(list)
for sid, t in bt.items():
    if sid in cond:
        states = set(s.get("state") for s in t.get("state_sequence", []))
        div[cond[sid]].append(len(states))
for c in ["seamless","sff"]:
    d = div[c]
    logger.info(f"  {c:8s} mean={sum(d)/len(d):.1f} distinct states")


In [ ]:
# Purpose: imports and small setup
import json, math
from collections import defaultdict, Counter

bt = {json.loads(l)["session_id"]: json.loads(l) for l in open(TAGGER_PATH) if l.strip()}
cond = {r["session_id"]: r["condition"] for r in batch_results if r["status"]=="ok"}

def seq_entropy(transitions):
    if not transitions: return 0.0
    c = Counter(transitions)
    tot = sum(c.values())
    return -sum((n/tot)*math.log2(n/tot) for n in c.values())

per_cond = defaultdict(lambda: {"entropies": [], "distinct_trans": [], "all_trans": Counter(), "repeat_ratio": []})

for sid, t in bt.items():
    if sid not in cond: continue
    states = [s.get("state") for s in t.get("state_sequence", [])]
    transitions = [(states[i], states[i+1]) for i in range(len(states)-1)]
    c = cond[sid]
    per_cond[c]["entropies"].append(seq_entropy(transitions))
    per_cond[c]["distinct_trans"].append(len(set(transitions)))
    per_cond[c]["all_trans"].update(transitions)
    # repeat ratio: fraction of transitions that are self-loops (X->X), a sign of spinning
    selfloops = sum(1 for a,b in transitions if a==b)
    per_cond[c]["repeat_ratio"].append(selfloops/len(transitions) if transitions else 0)

logger.info("=== TRANSITION STRUCTURE BY CONDITION (blinded tagger, batch_001) ===\n")
for c in ["seamless","sff"]:
    d = per_cond[c]
    n = len(d["entropies"])
    me = sum(d["entropies"])/n
    mdt = sum(d["distinct_trans"])/n
    mrr = sum(d["repeat_ratio"])/n
    logger.info(f"{c.upper()}  (n={n})")
    logger.info(f"  mean transition entropy : {me:.3f} bits")
    logger.info(f"  mean distinct transitions: {mdt:.1f}")
    logger.info(f"  mean self-loop ratio     : {mrr:.2f}  (higher = more spinning on same state)")
    logger.info(f"  top 5 transitions        : {[f'{a[0][:4]}>{a[1][:4]}:{n2}' for a,n2 in d['all_trans'].most_common(5)]}")
    logger.info()

# headline comparison
se, sf = per_cond["seamless"], per_cond["sff"]
se_ent = sum(se["entropies"])/len(se["entropies"])
sf_ent = sum(sf["entropies"])/len(sf["entropies"])
logger.info("="*55)
logger.info(f"H2 test — transition entropy: seamless={se_ent:.3f}  sff={sf_ent:.3f}  diff={sf_ent-se_ent:+.3f}")
logger.info(f"  -> {'SFF MORE diverse (H2 supported)' if sf_ent > se_ent else 'SFF NOT more diverse (H2 weak/unsupported)'}")


In [ ]:
# Purpose: imports and small setup
import json
rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
# grab one SFF session, show its last learner turn
sff_sids = [r["session_id"] for r in rows if "sff" in r["session_id"]]
sid = sff_sids[0]
turns = sorted([r for r in rows if r["session_id"]==sid], key=lambda x:x["turn_index"])
logger.info("Session:", sid)
for t in turns:
    logger.info(f"  t{t['turn_index']} [{t['speaker']}] {t['state_label']}: {t['visible_response'][:70]}")
last_learner = [t for t in turns if t["speaker"]=="learner"][-1]
logger.info("\nLast learner turn text (what Option A would judge):")
logger.info(last_learner["visible_response"][:300])


In [ ]:
# Purpose: imports and small setup
import json

rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
tasks = {json.loads(l)["session_id"]: json.loads(l) for l in open(TASKS_PATH) if l.strip()}
cond = {r["session_id"]: r["condition"] for r in batch_results if r["status"]=="ok"}

sessions = {}
for r in rows:
    sessions.setdefault(r["session_id"], []).append(r)

REJUDGE_PATH = RUN_DIR / "judge_scores_learner_target.jsonl"
from collections import defaultdict
new_scores = defaultdict(list)
n = 0

for sid, turns in sessions.items():
    turns = sorted(turns, key=lambda x: x["turn_index"])
    learner_turns = [t for t in turns if t["speaker"] == "learner"]
    if not learner_turns:
        continue
    final_learner_response = learner_turns[-1]["visible_response"]  # <-- Option A: judge the learner's last turn
    task = tasks.get(sid)
    if task is None:
        continue
    try:
        judge = run_judge(task=task, agent_response=final_learner_response, session_id=sid)
        rec = {"session_id": sid, "scored_target": "final_learner_turn", **judge["parsed"]}
        with REJUDGE_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        sc = (judge["parsed"].get("domain_raw_score") or {}).get("score_value")
        if sid in cond:
            new_scores[cond[sid]].append(sc)
        n += 1
        if n % 10 == 0:
            logger.info(f"  re-judged {n}/54...")
    except Exception as e:
        logger.info(f"  {sid[:40]} judge fail: {str(e)[:60]}")

logger.info(f"\n=== RE-JUDGE WITH LEARNER TARGET (Option A) ===")
logger.info(f"Re-judged {n} sessions\n")
for c in ["seamless", "sff"]:
    s = [x for x in new_scores[c] if x is not None]
    if s:
        dist = {v: s.count(v) for v in sorted(set(s))}
        logger.info(f"  {c:8s} n={len(s)} mean={sum(s)/len(s):.2f} dist={dist}")

logger.info("\n=== BEFORE vs AFTER (the evidence for Dacon) ===")
logger.info("  agent-final-turn target : seamless 3.00, sff 0.70")
logger.info(f"  learner-final-turn target: seamless {sum(x for x in new_scores['seamless'] if x is not None)/len([x for x in new_scores['seamless'] if x is not None]):.2f}, sff {sum(x for x in new_scores['sff'] if x is not None)/len([x for x in new_scores['sff'] if x is not None]):.2f}")


In [ ]:
# Purpose: imports and small setup
import json
rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
sessions = {}
for r in rows:
    sessions.setdefault(r["session_id"], []).append(r)

# look at state labels of turns, and which carry substantial content
from collections import Counter
state_lens = {}
for sid, turns in sessions.items():
    for t in turns:
        st = t["state_label"]
        ln = len(t["visible_response"])
        state_lens.setdefault(st, []).append(ln)

logger.info("Mean response length by state label (which states carry real answers):")
for st in sorted(state_lens):
    lens = state_lens[st]
    logger.info(f"  {st:24s} n={len(lens):3d} mean_len={sum(lens)//len(lens):4d} chars")

# how many candidate answer-turns per session if we judge S3/S5/S9 + any long turn
logger.info("\nCandidate answer-turns per session (S3_DIRECT_ANSWER, S5_USER_REVISION, S9, or >200 chars):")
counts = []
for sid, turns in sessions.items():
    cands = [t for t in turns if t["state_label"] in ("S3_DIRECT_ANSWER","S5_USER_REVISION","S9_TASK_COMPLETE") or len(t["visible_response"])>200]
    counts.append(len(cands))
logger.info(f"  mean {sum(counts)/len(counts):.1f} candidates/session, max {max(counts)}, min {min(counts)}")


In [ ]:
# Purpose: imports and small setup
import json
from collections import defaultdict

rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
tasks = {json.loads(l)["session_id"]: json.loads(l) for l in open(TASKS_PATH) if l.strip()}
cond = {r["session_id"]: r["condition"] for r in batch_results if r["status"]=="ok"}

sessions = {}
for r in rows:
    sessions.setdefault(r["session_id"], []).append(r)

# Candidate = solution-bearing turns: S3_DIRECT_ANSWER (agent answer) or S5_USER_REVISION (learner worked answer)
SOLUTION_STATES = {"S3_DIRECT_ANSWER", "S5_USER_REVISION"}

REJUDGE_B_PATH = RUN_DIR / "judge_scores_best_answer.jsonl"
new_scores = defaultdict(list)
n_sessions = 0
n_calls = 0

for sid, turns in sessions.items():
    turns = sorted(turns, key=lambda x: x["turn_index"])
    task = tasks.get(sid)
    if task is None:
        continue
    candidates = [t for t in turns if t["state_label"] in SOLUTION_STATES and len(t["visible_response"].strip()) > 50]
    # fallback: if no solution-bearing turn, judge the final turn (so every session gets a score)
    if not candidates:
        candidates = [turns[-1]]

    best_score = None
    best_record = None
    for cand in candidates:
        try:
            judge = run_judge(task=task, agent_response=cand["visible_response"], session_id=sid)
            n_calls += 1
            sc = (judge["parsed"].get("domain_raw_score") or {}).get("score_value")
            if sc is not None and (best_score is None or sc > best_score):
                best_score = sc
                best_record = {"session_id": sid, "scored_target": "best_answer",
                               "best_turn_index": cand["turn_index"], "best_turn_speaker": cand["speaker"],
                               "n_candidates": len(candidates), **judge["parsed"]}
        except Exception as e:
            logger.info(f"  {sid[:30]} t{cand['turn_index']} fail: {str(e)[:50]}")

    if best_record:
        with REJUDGE_B_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(best_record, ensure_ascii=False) + "\n")
        if sid in cond:
            new_scores[cond[sid]].append(best_score)
        n_sessions += 1
        if n_sessions % 10 == 0:
            logger.info(f"  {n_sessions} sessions done ({n_calls} judge calls so far)...")

logger.info(f"\n=== OPTION B: BEST-ANSWER-IN-SESSION (max over S3/S5 turns) ===")
logger.info(f"{n_sessions} sessions, {n_calls} judge calls\n")
for c in ["seamless", "sff"]:
    s = [x for x in new_scores[c] if x is not None]
    if s:
        dist = {v: s.count(v) for v in sorted(set(s))}
        logger.info(f"  {c:8s} n={len(s)} mean={sum(s)/len(s):.2f} dist={dist}")

logger.info("\n=== ALL THREE TARGETS COMPARED ===")
logger.info("  agent-final  : seamless 3.00, sff 0.70")
logger.info("  learner-final: seamless 1.68, sff 1.30")
sm = [x for x in new_scores['seamless'] if x is not None]
sf = [x for x in new_scores['sff'] if x is not None]
logger.info(f"  best-answer  : seamless {sum(sm)/len(sm):.2f}, sff {sum(sf)/len(sf):.2f}")


In [ ]:
# Purpose: imports and small setup
import json
rejudge_b = {json.loads(l)["session_id"]: json.loads(l) for l in open(REJUDGE_B_PATH) if l.strip()}
rows = [json.loads(l) for l in open(SESSIONS_PATH) if l.strip()]
sessions = {}
for r in rows:
    sessions.setdefault(r["session_id"], []).append(r)

logger.info("SFF sessions scoring 0 under best-answer — why?\n")
for sid, rec in rejudge_b.items():
    if "sff" not in sid: continue
    sc = (rec.get("domain_raw_score") or {}).get("score_value")
    if sc != 0: continue
    turns = sorted(sessions[sid], key=lambda x: x["turn_index"])
    had_solution = any(t["state_label"] in ("S3_DIRECT_ANSWER","S5_USER_REVISION") for t in turns)
    reached_terminal = turns[-1]["state_label"] in ("S9_TASK_COMPLETE","S10_FAILURE")
    logger.info(f"  {sid[:36]:36s} solution_turn={had_solution} terminal={reached_terminal} judged_turn=t{rec.get('best_turn_index')}({rec.get('best_turn_speaker')})")


In [ ]:
# Purpose: imports and small setup
import json, itertools, random, time
from pathlib import Path

# ---- Full run config ----
RUN_ID = "full_run_001"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH = RUN_DIR / "sessions.jsonl"
JUDGE_PATH = RUN_DIR / "judge_scores.jsonl"
TAGGER_PATH = RUN_DIR / "state_tagger.jsonl"
INVALID_PATH = RUN_DIR / "invalid_sessions.jsonl"
TASKS_PATH = RUN_DIR / "session_tasks.jsonl"

CONDITIONS = ["seamless", "sff"]
DOMAINS = ["python_debugging", "clinical_text_interpretation", "historical_document_synthesis"]
PERSONAS = ["novice", "intermediate", "expert"]
DIFFICULTIES = ["easy", "medium", "hard"]
REPLICATES = 100   # per the spec: 100 sessions per cell

# Build the full manifest with DETERMINISTIC ids (key for resume)
# Design: 2 cond x 3 domain x 3 persona = 18 cells, 100 each = 1800.
# Within each cell, rotate difficulty tiers across the 100 replicates.
manifest = []
for condition in CONDITIONS:
    for domain in DOMAINS:
        for persona in PERSONAS:
            for rep in range(REPLICATES):
                difficulty = DIFFICULTIES[rep % 3]  # rotate easy/med/hard
                sid = f"s_{condition}_{domain}_{persona}_{rep:03d}"
                manifest.append({
                    "session_id": sid, "condition": condition, "domain": domain,
                    "persona_level": persona, "difficulty": difficulty, "replicate": rep,
                })

logger.info(f"Full manifest: {len(manifest)} sessions")

# ---- Resume check: which sessions are already complete? ----
def completed_session_ids():
    done = set()
    if SESSIONS_PATH.exists():
        with open(SESSIONS_PATH) as f:
            for line in f:
                if line.strip():
                    done.add(json.loads(line)["session_id"])
    return done

already_done = completed_session_ids()
remaining = [m for m in manifest if m["session_id"] not in already_done]
logger.info(f"Already complete: {len(already_done)}")
logger.info(f"Remaining to run: {len(remaining)}")
logger.info(f"Logging to: {RUN_DIR}")


In [ ]:
# Purpose: defines function(s): run_chunk
def run_chunk(chunk_size=200, verbose_every=10):
    """Run up to chunk_size remaining sessions. Re-run to continue; skips completed."""
    random.seed(20260606)
    done = completed_session_ids()
    todo = [m for m in manifest if m["session_id"] not in done]
    if not todo:
        logger.info("All 1800 sessions complete.")
        return
    batch = todo[:chunk_size]
    logger.info(f"Running {len(batch)} sessions ({len(done)} already done, {len(todo)} remaining)\n")

    used_task_ids = set()
    # reload used tasks so we don't reuse across resumes
    if TASKS_PATH.exists():
        with open(TASKS_PATH) as f:
            for line in f:
                if line.strip():
                    used_task_ids.add(json.loads(line).get("task_id"))

    start = time.time()
    ok = fail = 0
    for i, m in enumerate(batch):
        sid = m["session_id"]
        domain = m["domain"]
        # pick a task of the right difficulty, not already used
        pool = TASK_POOLS[domain]
        cands = [t for t in pool if t["difficulty_tier"]==m["difficulty"] and t["task_id"] not in used_task_ids]
        if not cands:
            cands = [t for t in pool if t["task_id"] not in used_task_ids] or pool
        task = random.choice(cands)
        used_task_ids.add(task["task_id"])

        try:
            session = run_one_session(condition=m["condition"], persona_level=m["persona_level"], task=task, session_id=sid)
        except Exception as e:
            with INVALID_PATH.open("a") as f:
                f.write(json.dumps({"session_id": sid, "reason": f"crash: {type(e).__name__}: {e}"})+"\n")
            fail += 1
            continue

        # write task record + turns FIRST (this is what resume checks)
        with TASKS_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"session_id": sid, **task}, ensure_ascii=False)+"\n")
        with SESSIONS_PATH.open("a", encoding="utf-8") as f:
            for turn in session["turns"]:
                f.write(json.dumps({"session_id": sid, **turn}, ensure_ascii=False)+"\n")

        # tagger
        prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
        try:
            tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=sid)
            with TAGGER_PATH.open("a", encoding="utf-8") as f:
                f.write(json.dumps({"session_id": sid, **tagger["parsed"]}, ensure_ascii=False)+"\n")
        except Exception as e:
            pass  # tagger can be re-run later from transcripts

        # judge (best-answer target, exploratory) -- score max over S3/S5 turns
        try:
            turns = sorted(session["turns"], key=lambda x:x["turn_index"])
            cands_turns = [t for t in turns if t["state_label"] in ("S3_DIRECT_ANSWER","S5_USER_REVISION") and len(t["visible_response"].strip())>50] or [turns[-1]]
            best = None
            for ct in cands_turns:
                j = run_judge(task=task, agent_response=ct["visible_response"], session_id=sid)
                sc = (j["parsed"].get("domain_raw_score") or {}).get("score_value")
                if best is None or (sc is not None and sc > (best.get("domain_raw_score",{}) or {}).get("score_value",-1)):
                    best = j["parsed"]
            if best:
                with JUDGE_PATH.open("a", encoding="utf-8") as f:
                    f.write(json.dumps({"session_id": sid, "scored_target":"best_answer", **best}, ensure_ascii=False)+"\n")
        except Exception as e:
            pass  # judge can be re-run later

        ok += 1
        if (i+1) % verbose_every == 0:
            el = time.time()-start
            rate = (i+1)/el
            eta = (len(batch)-(i+1))/rate/60
            logger.info(f"  {i+1}/{len(batch)}  ok={ok} fail={fail}  {rate*60:.0f}/min  chunk ETA {eta:.0f}min")

    el = time.time()-start
    total_done = len(completed_session_ids())
    logger.info(f"\nChunk done: {ok} ok, {fail} failed, {el/60:.1f} min. Total complete: {total_done}/1800")
    logger.info(f"Re-run run_chunk() to continue." if total_done < 1800 else "FULL RUN COMPLETE.")


In [ ]:
# Purpose: data processing / analysis step
logger.info("run_chunk" in dir())


In [ ]:
# Purpose: imports and small setup
import subprocess
result = subprocess.run(
    ["python", str(PROJECT_ROOT / "scripts" / "download_benchmark_candidates.py"), "--help"],
    capture_output=True, text=True
)
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')


In [ ]:
# Purpose: imports and small setup
import subprocess
result = subprocess.run(
    ["python", str(PROJECT_ROOT / "scripts" / "download_benchmark_candidates.py"), "--include-large"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')
logger.info("=== STDERR ===")
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')


In [ ]:
# Purpose: imports and small setup
import subprocess, os
# locate the python pool builder
scripts = os.listdir(PROJECT_ROOT / "scripts")
logger.info("Scripts available:")
for s in scripts:
    if s.endswith(".py"):
        logger.info(" ", s)


In [ ]:
# Purpose: imports and small setup
import subprocess
result = subprocess.run(
    ["python", str(PROJECT_ROOT / "scripts" / "build_python_task_pool.py"), "--help"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')


In [ ]:
# Purpose: imports and small setup
import json
from pathlib import Path

# reload python pool from the rebuilt selected files
py_dir = PROJECT_ROOT / "python_domain" / "selected"
py_tasks = []
for f in py_dir.glob("*.jsonl"):
    with open(f) as fh:
        for line in fh:
            if line.strip():
                py_tasks.append(json.loads(line))

TASK_POOLS["python_debugging"] = py_tasks

# verify: count, benchmarks, tiers
from collections import Counter
logger.info(f"python_debugging now: {len(py_tasks)} tasks")
logger.info("by benchmark:", dict(Counter(t["benchmark_source"] for t in py_tasks)))
logger.info("by tier:", dict(Counter(t["difficulty_tier"] for t in py_tasks)))

# check APPS task prompt lengths (they can be very long - token cost concern)
apps_lens = [len(t["task_prompt"]) for t in py_tasks if t["benchmark_source"]=="APPS"]
if apps_lens:
    logger.info(f"\nAPPS prompt lengths: min={min(apps_lens)} max={max(apps_lens)} mean={sum(apps_lens)//len(apps_lens)} chars")
    over = sum(1 for l in apps_lens if l > 4000)
    logger.info(f"APPS tasks over 4000 chars: {over}/{len(apps_lens)}")


In [ ]:
# Purpose: data processing / analysis step
py_dir = PROJECT_ROOT / "python_domain" / "selected"
logger.info("Files in python_domain/selected/:")
for f in sorted(py_dir.glob("*")):
    n = sum(1 for line in open(f) if line.strip()) if f.suffix==".jsonl" else "-"
    logger.info(f"  {f.name}: {n} rows")


In [ ]:
# Purpose: imports and small setup
import json
from collections import Counter

py_dir = PROJECT_ROOT / "python_domain" / "selected"
canonical = ["mbpp_selected_60.jsonl", "humaneval_selected_60.jsonl", "apps_selected_60.jsonl"]
py_tasks = []
for fname in canonical:
    with open(py_dir / fname) as fh:
        for line in fh:
            if line.strip():
                py_tasks.append(json.loads(line))

TASK_POOLS["python_debugging"] = py_tasks
logger.info(f"python_debugging: {len(py_tasks)} tasks")
logger.info("by benchmark:", dict(Counter(t["benchmark_source"] for t in py_tasks)))
logger.info("by tier:", dict(Counter(t["difficulty_tier"] for t in py_tasks)))


In [ ]:
# Purpose: imports and small setup
from collections import Counter
for d in ["python_debugging", "clinical_text_interpretation", "historical_document_synthesis"]:
    pool = TASK_POOLS[d]
    benches = dict(Counter(t["benchmark_source"] for t in pool))
    tiers = dict(Counter(t["difficulty_tier"] for t in pool))
    # check for duplicate task_ids (the tell for double-loading)
    ids = [t["task_id"] for t in pool]
    dupes = len(ids) - len(set(ids))
    logger.info(f"{d}: {len(pool)} tasks, dupes={dupes}")
    logger.info(f"   benchmarks: {benches}")
    logger.info(f"   tiers: {tiers}")


In [ ]:
# Purpose: data processing / analysis step
run_chunk(20)


In [ ]:
# Purpose: imports and small setup
import json
for name, path in [("sessions", SESSIONS_PATH), ("tasks", TASKS_PATH), ("judge", JUDGE_PATH), ("tagger", TAGGER_PATH)]:
    if path.exists():
        sids = set(json.loads(l)["session_id"] for l in open(path) if l.strip())
        logger.info(f"{name:10s}: {len(sids)} unique sessions")
    else:
        logger.info(f"{name:10s}: FILE MISSING")


In [ ]:
# Purpose: imports and small setup
import time
while True:
    done = len(completed_session_ids())
    if done >= 1800:
        logger.info(f"ALL DONE: {done}/1800")
        break
    logger.info(f"\n=== {done}/1800 done — starting next chunk ===")
    try:
        run_chunk(200)
    except Exception as e:
        logger.info(f"Chunk errored: {type(e).__name__}: {e} — waiting 60s, retrying")
        time.sleep(60)


In [ ]:
# Purpose: data processing / analysis step
logger.info("run_chunk" in dir(), "TASK_POOLS" in dir(), "groq_client" in dir(), "completed_session_ids" in dir())


In [5]:
# Purpose: imports and small setup
import json
from pathlib import Path
# (sanitized path) p = Path('PROJECT_ROOT
if p.exists():
    sids = set(json.loads(l)["session_id"] for l in open(p) if l.strip())
    logger.info("SESSIONS SAVED:", len(sids))
else:
    logger.info("checking path...")


SESSIONS SAVED: 1317


In [6]:
# Purpose: data processing / analysis step
logger.info("run_chunk:", "run_chunk" in dir())
logger.info("TASK_POOLS:", "TASK_POOLS" in dir())
logger.info("groq_client:", "groq_client" in dir())
logger.info("completed_session_ids:", "completed_session_ids" in dir())


run_chunk: False
TASK_POOLS: True
groq_client: False
completed_session_ids: False


In [9]:
# Purpose: data processing / analysis step
for n in ["render_prompt","call_anthropic","call_openai","call_groq","call_with_json",
          "extract_json","run_persona_turn","run_judge","run_state_tagger",
          "run_agent_turn","run_one_session","normalize_state_label",
          "PROMPTS","TASK_POOLS","PROJECT_ROOT","VALID_STATES","TERMINAL_STATES",
          "anthropic_client","openai_client","AGENT_REQUIRED_KEYS","JUDGE_REQUIRED_KEYS",
          "STATE_TAGGER_REQUIRED_KEYS","PROMPT_FILES","TURN_CAP","MAX_RETRIES",
          "MODEL_HAIKU","MODEL_LLAMA","MODEL_PERSONA"]:
    logger.info(f"  {'OK ' if n in dir() else 'MISSING'} {n}")


  OK  render_prompt
  MISSING call_anthropic
  MISSING call_openai
  MISSING call_groq
  MISSING call_with_json
  MISSING extract_json
  MISSING run_persona_turn
  MISSING run_judge
  MISSING run_state_tagger
  MISSING run_agent_turn
  MISSING run_one_session
  MISSING normalize_state_label
  OK  PROMPTS
  OK  TASK_POOLS
  OK  PROJECT_ROOT
  MISSING VALID_STATES
  MISSING TERMINAL_STATES
  MISSING anthropic_client
  MISSING openai_client
  MISSING AGENT_REQUIRED_KEYS
  MISSING JUDGE_REQUIRED_KEYS
  MISSING STATE_TAGGER_REQUIRED_KEYS
  OK  PROMPT_FILES
  MISSING TURN_CAP
  MISSING MAX_RETRIES
  MISSING MODEL_HAIKU
  MISSING MODEL_LLAMA
  MISSING MODEL_PERSONA


In [11]:
# Purpose: execution-based scoring of code solutions
# ============ CONSOLIDATED KERNEL REBUILD (real source from notebook 03) ============
from google.colab import drive, userdata
from pathlib import Path
import sys, json, re, time, random, uuid, itertools
import anthropic, openai, groq

# --- Cell 0: paths + keys ---
# (sanitized path) PROJECT_ROOT = Path("PROJECT_ROOT")
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
PROMPTS_DIR = PROJECT_ROOT / "prompts"; CONFIGS_DIR = PROJECT_ROOT / "configs"
ANTHROPIC_KEY = userdata.get("Anthropic"); OPENAI_KEY = userdata.get("OPENAI_API_KEY"); GROQ_KEY = userdata.get("Groq")

# --- Cell 3: clients + call_* ---
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
openai_client = openai.OpenAI(api_key=OPENAI_KEY)
groq_client = groq.Groq(api_key=GROQ_KEY)
MODEL_HAIKU = "claude-haiku-4-5"; MODEL_GPT35 = "gpt-3.5-turbo"; MODEL_LLAMA = "llama-3.3-70b-versatile"
MAX_RETRIES = 4; BASE_BACKOFF = 1.5
ANTHROPIC_RETRYABLE = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError)
OPENAI_RETRYABLE = (openai.RateLimitError, openai.APIConnectionError, openai.InternalServerError)
GROQ_RETRYABLE = (groq.RateLimitError, groq.APIConnectionError, groq.InternalServerError)
def _backoff(attempt): return (BASE_BACKOFF ** attempt) + random.uniform(0, 0.5)
def call_anthropic(system, user, max_tokens=1500):
    for attempt in range(MAX_RETRIES):
        try:
            r = anthropic_client.messages.create(model=MODEL_HAIKU, max_tokens=max_tokens, system=system, messages=[{"role":"user","content":user}])
            return {"text": r.content[0].text, "tokens_in": r.usage.input_tokens, "tokens_out": r.usage.output_tokens, "model": MODEL_HAIKU, "stop_reason": r.stop_reason}
        except ANTHROPIC_RETRYABLE:
            if attempt == MAX_RETRIES-1: raise
            time.sleep(_backoff(attempt))
def call_openai(system, user, max_tokens=1500):
    for attempt in range(MAX_RETRIES):
        try:
            r = openai_client.chat.completions.create(model=MODEL_GPT35, max_tokens=max_tokens, messages=[{"role":"system","content":system},{"role":"user","content":user}])
            return {"text": r.choices[0].message.content, "tokens_in": r.usage.prompt_tokens, "tokens_out": r.usage.completion_tokens, "model": MODEL_GPT35, "stop_reason": r.choices[0].finish_reason}
        except OPENAI_RETRYABLE:
            if attempt == MAX_RETRIES-1: raise
            time.sleep(_backoff(attempt))
def call_groq(system, user, max_tokens=1500):
    for attempt in range(MAX_RETRIES):
        try:
            r = groq_client.chat.completions.create(model=MODEL_LLAMA, max_tokens=max_tokens, temperature=0.0, messages=[{"role":"system","content":system},{"role":"user","content":user}])
            return {"text": r.choices[0].message.content, "tokens_in": r.usage.prompt_tokens, "tokens_out": r.usage.completion_tokens, "model": MODEL_LLAMA, "stop_reason": r.choices[0].finish_reason}
        except GROQ_RETRYABLE:
            if attempt == MAX_RETRIES-1: raise
            time.sleep(_backoff(attempt))

# --- Cell 4: extract_json + call_with_json ---
JSON_FENCE_RE = re.compile(r"```(?:json)?\s*(\{.*?\})\s*```", re.DOTALL)
JSON_OBJECT_RE = re.compile(r"(\{.*\})", re.DOTALL)
def extract_json(text):
    text = text.strip()
    try: return json.loads(text)
    except json.JSONDecodeError: pass
    m = JSON_FENCE_RE.search(text)
    if m:
        try: return json.loads(m.group(1))
        except json.JSONDecodeError: pass
    m = JSON_OBJECT_RE.search(text)
    if m:
        try: return json.loads(m.group(1))
        except json.JSONDecodeError: pass
    raise ValueError(f"No valid JSON object found. First 300: {text[:300]!r}")
def call_with_json(call_fn, system, user, max_tokens=1500, required_keys=None, max_attempts=3):
    last_error=last_raw=None
    for attempt in range(max_attempts):
        response = call_fn(system=system, user=user, max_tokens=max_tokens)
        last_raw = response["text"]
        try: parsed = extract_json(last_raw)
        except ValueError as e: last_error=str(e); continue
        if required_keys and not required_keys.issubset(parsed.keys()):
            last_error=f"Missing keys: {sorted(required_keys-set(parsed.keys()))}"; continue
        return {"parsed":parsed,"raw_text":last_raw,"tokens_in":response["tokens_in"],"tokens_out":response["tokens_out"],"model":response["model"],"stop_reason":response["stop_reason"],"attempts":attempt+1}
    raise RuntimeError(f"JSON extraction failed after {max_attempts} attempts. Last error: {last_error}. Raw: {last_raw[:400] if last_raw else None!r}")

# --- Cell 2: render_prompt + PROMPTS reload ---
PROMPT_REQUIRED_VARS = {
    "seamless_agent": {"session_id","domain","persona_level","difficulty_tier","benchmark_source","benchmark_task_id","task_prompt","task_context","prior_turns"},
    "sff_agent": {"session_id","domain","persona_level","difficulty_tier","benchmark_source","benchmark_task_id","task_prompt","task_context","prior_turns"},
    "persona_novice": {"session_id","domain","difficulty_tier","benchmark_source","benchmark_task_id","prior_turns"},
    "persona_intermediate": {"session_id","domain","difficulty_tier","benchmark_source","benchmark_task_id","prior_turns"},
    "persona_expert": {"session_id","domain","difficulty_tier","benchmark_source","benchmark_task_id","prior_turns"},
    "judge": {"session_id","domain","difficulty_tier","benchmark_source","benchmark_task_id","task_prompt","ground_truth","frozen_rubric","required_output_schema","test_pass_rate","local_execution_logs","agent_response"},
    "state_tagger": {"session_id","domain","difficulty_tier","prior_turns"},
}
PLACEHOLDER_RE = re.compile(r"\{\{([a-z_]+)\}\}")
def render_prompt(prompt_name, variables):
    if prompt_name not in PROMPTS: raise KeyError(f"Unknown prompt: {prompt_name}")
    template = PROMPTS[prompt_name]; required = PROMPT_REQUIRED_VARS[prompt_name]
    missing = required - set(variables.keys())
    if missing: raise ValueError(f"Missing variables for {prompt_name}: {sorted(missing)}")
    found = set(PLACEHOLDER_RE.findall(template))
    unexpected = required - found
    if unexpected: raise ValueError(f"{prompt_name} declares vars not in template: {sorted(unexpected)}")
    def stringify(v): return v if isinstance(v,str) else json.dumps(v, ensure_ascii=False, indent=2)
    rendered = template
    for var in found:
        if var in variables: rendered = rendered.replace(f"{{{{{var}}}}}", stringify(variables[var]))
    remaining = PLACEHOLDER_RE.findall(rendered)
    if remaining: raise ValueError(f"Unfilled placeholders in {prompt_name}: {sorted(set(remaining))}")
    return rendered

logger.info("PROMPTS loaded?", "PROMPTS" in dir(), "| TASK_POOLS loaded?", "TASK_POOLS" in dir())


PROMPTS loaded? True | TASK_POOLS loaded? True


In [12]:
# Purpose: load run/session data from disk
# ============ PART 2: states, role fns, session, full_run paths, run_chunk ============

# --- Cell 24: VALID_STATES + normalize_state_label ---
VALID_STATES = {
    "S0_START","S1_TASK_PRESENTED","S2_INITIAL_ORIENTATION","S3_DIRECT_ANSWER",
    "S4_SOCRATIC_PROMPT","S5_USER_REVISION","S6_USER_ACCEPT","S7_USER_REJECT",
    "S8_FEEDBACK_LOOP","S9_TASK_COMPLETE","S10_FAILURE",
}
def normalize_state_label(raw_label):
    if raw_label in VALID_STATES: return raw_label, True
    return None, False

# --- Cell 5: required-keys + persona map + role functions ---
AGENT_REQUIRED_KEYS = {"state_label","visible_response","policy_action"}
PERSONA_REQUIRED_KEYS = {"state_label","visible_response","policy_action"}
JUDGE_REQUIRED_KEYS = {"session_id","domain_raw_score"}
STATE_TAGGER_REQUIRED_KEYS = {"state_sequence","terminal_state"}
PERSONA_PROMPT_BY_LEVEL = {"novice":"persona_novice","intermediate":"persona_intermediate","expert":"persona_expert"}

# --- Cell 33: corrected run_agent_turn (WITH prior_turns) ---
def run_agent_turn(condition, task, persona_level, session_id, prior_turns=None, task_context=""):
    prompt_name = "sff_agent" if condition == "sff" else "seamless_agent"
    variables = {
        "session_id": session_id, "domain": task["domain"], "persona_level": persona_level,
        "difficulty_tier": task["difficulty_tier"], "benchmark_source": task["benchmark_source"],
        "benchmark_task_id": task["task_id"], "task_prompt": task["task_prompt"],
        "task_context": task_context or task.get("context","") or "",
        "prior_turns": prior_turns if prior_turns is not None else [],
    }
    rendered = render_prompt(prompt_name, variables)
    return call_with_json(call_anthropic, system=rendered, user="Produce the next agent turn following the system policy. Output JSON only.", max_tokens=1500, required_keys=AGENT_REQUIRED_KEYS)

def run_persona_turn(persona_level, task, prior_turns, session_id):
    prompt_name = PERSONA_PROMPT_BY_LEVEL[persona_level]
    variables = {
        "session_id": session_id, "domain": task["domain"], "difficulty_tier": task["difficulty_tier"],
        "benchmark_source": task["benchmark_source"], "benchmark_task_id": task["task_id"], "prior_turns": prior_turns,
    }
    rendered = render_prompt(prompt_name, variables)
    return call_with_json(call_openai, system=rendered, user="Produce the next learner turn following the system policy. Output JSON only.", max_tokens=1000, required_keys=PERSONA_REQUIRED_KEYS)

def run_judge(task, agent_response, session_id, test_pass_rate=None, local_execution_logs=""):
    variables = {
        "session_id": session_id, "domain": task["domain"], "difficulty_tier": task["difficulty_tier"],
        "benchmark_source": task["benchmark_source"], "benchmark_task_id": task["task_id"],
        "task_prompt": task["task_prompt"], "ground_truth": task["ground_truth"],
        "frozen_rubric": task.get("rubric",""), "required_output_schema": task.get("task_type",""),
        "test_pass_rate": test_pass_rate if test_pass_rate is not None else "not_applicable",
        "local_execution_logs": local_execution_logs or "not_applicable", "agent_response": agent_response,
    }
    rendered = render_prompt("judge", variables)
    return call_with_json(call_groq, system=rendered, user="Audit the final session and produce the blinded score. Output JSON only.", max_tokens=2000, required_keys=JUDGE_REQUIRED_KEYS)

def run_state_tagger(task, prior_turns, session_id):
    variables = {"session_id": session_id, "domain": task["domain"], "difficulty_tier": task["difficulty_tier"], "prior_turns": prior_turns}
    rendered = render_prompt("state_tagger", variables)
    return call_with_json(call_groq, system=rendered, user="Re-tag the session blind. Output JSON only.", max_tokens=2000, required_keys=STATE_TAGGER_REQUIRED_KEYS)

# --- Cell 7/36: TURN_CAP + corrected run_one_session (guardrail + history) ---
TURN_CAP = 8
TERMINAL_STATES = {"S9_TASK_COMPLETE","S10_FAILURE"}
def run_one_session(condition, persona_level, task, session_id):
    turns=[]; prior_turns_view=[]; total_tokens_in=total_tokens_out=0; failure_reason=None
    for turn_index in range(TURN_CAP):
        try:
            if turn_index % 2 == 0:
                role="agent"; result=run_agent_turn(condition=condition, task=task, persona_level=persona_level, session_id=session_id, prior_turns=prior_turns_view)
            else:
                role="learner"; result=run_persona_turn(persona_level=persona_level, task=task, prior_turns=prior_turns_view, session_id=session_id)
        except Exception as e:
            failure_reason=f"{role}_call_failed_turn_{turn_index}: {type(e).__name__}: {e}"; break
        parsed=result["parsed"]; raw_state=parsed.get("state_label")
        clean_state, state_valid = normalize_state_label(raw_state)
        turns.append({"turn_index":turn_index,"speaker":role,"state_label":raw_state,"state_label_valid":state_valid,
            "policy_action":parsed.get("policy_action"),"visible_response":parsed.get("visible_response",""),
            "metadata":parsed.get("metadata",{}),"model":result["model"],"tokens_in":result["tokens_in"],
            "tokens_out":result["tokens_out"],"attempts":result["attempts"],"raw_text":result["raw_text"]})
        prior_turns_view.append({"turn_index":turn_index,"speaker":role,"visible_response":parsed.get("visible_response","")})
        total_tokens_in+=result["tokens_in"]; total_tokens_out+=result["tokens_out"]
        if state_valid and clean_state in TERMINAL_STATES: break
    final_agent_response = next((t["visible_response"] for t in reversed(turns) if t["speaker"]=="agent"), "")
    invalid_label_count = sum(1 for t in turns if not t["state_label_valid"])
    return {"session_id":session_id,"condition":condition,"persona_level":persona_level,"domain":task["domain"],
        "difficulty_tier":task["difficulty_tier"],"benchmark_source":task["benchmark_source"],"benchmark_task_id":task["task_id"],
        "turns":turns,"turn_count":len(turns),"final_agent_response":final_agent_response,
        "total_tokens_in":total_tokens_in,"total_tokens_out":total_tokens_out,"invalid_label_count":invalid_label_count,
        "hit_turn_cap":len(turns)==TURN_CAP and (not turns or not (turns[-1]["state_label_valid"] and normalize_state_label(turns[-1]["state_label"])[0] in TERMINAL_STATES)),
        "failure_reason":failure_reason}

# --- Cell 85: full_run_001 paths + manifest + completed_session_ids ---
RUN_ID="full_run_001"; RUN_DIR=PROJECT_ROOT/"runs"/RUN_ID; RUN_DIR.mkdir(parents=True, exist_ok=True)
SESSIONS_PATH=RUN_DIR/"sessions.jsonl"; JUDGE_PATH=RUN_DIR/"judge_scores.jsonl"
TAGGER_PATH=RUN_DIR/"state_tagger.jsonl"; INVALID_PATH=RUN_DIR/"invalid_sessions.jsonl"; TASKS_PATH=RUN_DIR/"session_tasks.jsonl"
CONDITIONS=["seamless","sff"]; DOMAINS=["python_debugging","clinical_text_interpretation","historical_document_synthesis"]
PERSONAS=["novice","intermediate","expert"]; DIFFICULTIES=["easy","medium","hard"]; REPLICATES=100
manifest=[]
for condition in CONDITIONS:
    for domain in DOMAINS:
        for persona in PERSONAS:
            for rep in range(REPLICATES):
                manifest.append({"session_id":f"s_{condition}_{domain}_{persona}_{rep:03d}","condition":condition,
                    "domain":domain,"persona_level":persona,"difficulty":DIFFICULTIES[rep%3],"replicate":rep})
def completed_session_ids():
    done=set()
    if SESSIONS_PATH.exists():
        with open(SESSIONS_PATH) as f:
            for line in f:
                if line.strip(): done.add(json.loads(line)["session_id"])
    return done

logger.info("Manifest:", len(manifest), "| Completed so far:", len(completed_session_ids()))


Manifest: 1800 | Completed so far: 1317


In [14]:
# Purpose: defines function(s): run_chunk
def run_chunk(chunk_size=200, verbose_every=10):
    random.seed(20260606)
    done = completed_session_ids()
    todo = [m for m in manifest if m["session_id"] not in done]
    if not todo:
        logger.info("All 1800 sessions complete."); return
    batch = todo[:chunk_size]
    logger.info(f"Running {len(batch)} sessions ({len(done)} already done, {len(todo)} remaining)\n")
    used_task_ids = set()
    if TASKS_PATH.exists():
        with open(TASKS_PATH) as f:
            for line in f:
                if line.strip(): used_task_ids.add(json.loads(line).get("task_id"))
    start = time.time(); ok = fail = 0
    for i, m in enumerate(batch):
        sid = m["session_id"]; domain = m["domain"]; pool = TASK_POOLS[domain]
        cands = [t for t in pool if t["difficulty_tier"]==m["difficulty"] and t["task_id"] not in used_task_ids]
        if not cands:
            cands = [t for t in pool if t["task_id"] not in used_task_ids] or pool
        task = random.choice(cands); used_task_ids.add(task["task_id"])
        try:
            session = run_one_session(condition=m["condition"], persona_level=m["persona_level"], task=task, session_id=sid)
        except Exception as e:
            with INVALID_PATH.open("a") as f:
                f.write(json.dumps({"session_id": sid, "reason": f"crash: {type(e).__name__}: {e}"})+"\n")
            fail += 1; continue
        with TASKS_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"session_id": sid, **task}, ensure_ascii=False)+"\n")
        with SESSIONS_PATH.open("a", encoding="utf-8") as f:
            for turn in session["turns"]:
                f.write(json.dumps({"session_id": sid, **turn}, ensure_ascii=False)+"\n")
        prior_view = [{"turn_index": t["turn_index"], "speaker": t["speaker"], "visible_response": t["visible_response"]} for t in session["turns"]]
        try:
            tagger = run_state_tagger(task=task, prior_turns=prior_view, session_id=sid)
            with TAGGER_PATH.open("a", encoding="utf-8") as f:
                f.write(json.dumps({"session_id": sid, **tagger["parsed"]}, ensure_ascii=False)+"\n")
        except Exception: pass
        try:
            turns = sorted(session["turns"], key=lambda x:x["turn_index"])
            cands_turns = [t for t in turns if t["state_label"] in ("S3_DIRECT_ANSWER","S5_USER_REVISION") and len(t["visible_response"].strip())>50] or [turns[-1]]
            best = None
            for ct in cands_turns:
                j = run_judge(task=task, agent_response=ct["visible_response"], session_id=sid)
                sc = (j["parsed"].get("domain_raw_score") or {}).get("score_value")
                if best is None or (sc is not None and sc > (best.get("domain_raw_score",{}) or {}).get("score_value",-1)):
                    best = j["parsed"]
            if best:
                with JUDGE_PATH.open("a", encoding="utf-8") as f:
                    f.write(json.dumps({"session_id": sid, "scored_target":"best_answer", **best}, ensure_ascii=False)+"\n")
        except Exception: pass
        ok += 1
        if (i+1) % verbose_every == 0:
            el = time.time()-start; rate = (i+1)/el; eta = (len(batch)-(i+1))/rate/60
            logger.info(f"  {i+1}/{len(batch)}  ok={ok} fail={fail}  {rate*60:.0f}/min  chunk ETA {eta:.0f}min")
    el = time.time()-start; total_done = len(completed_session_ids())
    logger.info(f"\nChunk done: {ok} ok, {fail} failed, {el/60:.1f} min. Total complete: {total_done}/1800")
    logger.info("Re-run run_chunk() to continue." if total_done < 1800 else "FULL RUN COMPLETE.")

logger.info("run_chunk defined:", "run_chunk" in dir())


run_chunk defined: True


In [15]:
# Purpose: imports and small setup
import time
while True:
    done = len(completed_session_ids())
    if done >= 1800:
        logger.info(f"ALL DONE: {done}/1800"); break
    logger.info(f"\n=== {done}/1800 done — starting next chunk ===")
    try:
        run_chunk(200)
    except Exception as e:
        logger.info(f"Chunk errored: {type(e).__name__}: {e} — waiting 60s, retrying")
        time.sleep(60)



=== 1317/1800 done — starting next chunk ===
Running 200 sessions (1317 already done, 483 remaining)

  10/200  ok=10 fail=0  2/min  chunk ETA 88min
  20/200  ok=20 fail=0  2/min  chunk ETA 81min
  30/200  ok=30 fail=0  2/min  chunk ETA 76min
  40/200  ok=40 fail=0  2/min  chunk ETA 71min
  50/200  ok=50 fail=0  2/min  chunk ETA 66min
  60/200  ok=60 fail=0  2/min  chunk ETA 61min
  70/200  ok=70 fail=0  2/min  chunk ETA 56min
  80/200  ok=80 fail=0  2/min  chunk ETA 52min
  90/200  ok=90 fail=0  2/min  chunk ETA 47min
  100/200  ok=100 fail=0  2/min  chunk ETA 43min
  110/200  ok=110 fail=0  2/min  chunk ETA 39min
  120/200  ok=120 fail=0  2/min  chunk ETA 34min
  130/200  ok=130 fail=0  2/min  chunk ETA 30min
  140/200  ok=140 fail=0  2/min  chunk ETA 26min
  150/200  ok=150 fail=0  2/min  chunk ETA 22min
  160/200  ok=160 fail=0  2/min  chunk ETA 17min
  170/200  ok=170 fail=0  2/min  chunk ETA 13min
  180/200  ok=180 fail=0  2/min  chunk ETA 9min
  190/200  ok=190 fail=0  2/min  c

In [16]:
# Purpose: imports and small setup
import json
from pathlib import Path
# (sanitized path) RUN_DIR = Path("PROJECT_ROOT")
for name in ["sessions","session_tasks","judge_scores","state_tagger"]:
    p = RUN_DIR / f"{name}.jsonl"
    sids = set(json.loads(l)["session_id"] for l in open(p) if l.strip())
    logger.info(f"{name:16s}: {len(sids)} unique sessions")
inv = RUN_DIR / "invalid_sessions.jsonl"
logger.info(f"invalid_sessions: {sum(1 for l in open(inv) if l.strip()) if inv.exists() else 0} crashes")


sessions        : 1800 unique sessions
session_tasks   : 1800 unique sessions
judge_scores    : 1797 unique sessions
state_tagger    : 1799 unique sessions
invalid_sessions: 0 crashes


In [17]:
# Purpose: imports and small setup
from pathlib import Path
# (sanitized path) RUN_DIR = Path("PROJECT_ROOT")
for f in sorted(RUN_DIR.glob("*")):
    logger.info(f"{f.name:30s} {f.stat().st_size/1024:.0f} KB")


judge_scores.jsonl             2825 KB
session_tasks.jsonl            7198 KB
sessions.jsonl                 28607 KB
state_tagger.jsonl             2640 KB


In [ ]:
# Purpose: empty cell
